#### Notes on environment activation
source .venv/bin/activate

# Reinforcement Learning project:
## Adaptive optics control with reinforcement learning

In [1]:
!pip install hcipy
!pip install --force-reinstall xxhash==3.8.1

  Using cached xxhash-3.8.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (15 kB)
Using cached xxhash-3.8.1-cp313-cp313-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (220 kB)
  Attempting uninstall: xxhash
    Found existing installation: xxhash 3.8.1
    Uninstalling xxhash-3.8.1:
      Successfully uninstalled xxhash-3.8.1


In [2]:
import hcipy as hc
import xxhash as xx
import numpy as np
import matplotlib.pyplot as plt
import scipy.ndimage as ndimage
import time
import os
from tqdm.notebook import tqdm

print(hc.__version__)
print(xx.VERSION)

0.7.1
3.8.1


### General parameters

In [3]:
# wavelength_wfs = 842.0E-9 #nm
# telescope_diameter = 6.5 #m
# # zero_magnitude_flux = 3.9E10
# # stellar_magnitude = 0

### Pyramid Wavefront Sensor

In [4]:
# num_pupil_pixels = 70
# pupil_grid_diameter = 60/56 * telescope_diameter

# #input pupil grid for the P-WFS
# pupil_grid = hc.make_pupil_grid(num_pupil_pixels, pupil_grid_diameter)
# magellan_aperture = hc.evaluate_supersampled(hc.make_magellan_aperture(), pupil_grid, 6, make_sparse = False) # mak_magellan_aperture() if no arguments telescope diameter is 6.5m. By changing the diameter of the pupil_grid the proportion grid-aperture changes.

# # #show input pupil
# # hc.imshow_field(magellan_aperture)
# # plt.xlabel('x position(m)')
# # plt.ylabel('y position(m)')
# # plt.colorbar()
# # plt.show()


In [5]:
# # output pupil grid for P-WFS
# pwfs_grid = hc.make_pupil_grid(140, 2 * pupil_grid_diameter)
# pwfs = hc.PyramidWavefrontSensorOptics(pupil_grid, pwfs_grid, separation=pupil_grid_diameter, pupil_diameter=telescope_diameter, wavelength_0=wavelength_wfs, q=3)
# camera = hc.NoiselessDetector(pwfs_grid)
# wf = hc.Wavefront(magellan_aperture, wavelength_wfs)
# wf.total_power = 1

# camera.integrate(pwfs.forward(wf), 1)

# image_ref = camera.read_out()
# image_ref /= image_ref.sum()

# hc.imshow_field(image_ref)
# plt.colorbar()
# plt.show()

In [6]:
# def pwfs_mask(image, threshold=1.5e-5):
#     return image > threshold

# mask = pwfs_mask(image_ref)
# hc.imshow_field(mask)
# plt.colorbar()
# plt.show()

In [7]:
# from scipy import ndimage

# def build_pwfs_mask(image_ref, pwfs_grid, threshold=0.5e-5, smooth_sigma=2, fill_holes=True):
#     """
#     Costruisce una maschera robusta per le 4 pupille del PWFS non modulato,
#     a partire dall'immagine di riferimento gia' calcolata in AOS.__init__
#     (self.image_ref).

#     Parametri
#     ----------
#     image_ref : Field
#         Immagine di riferimento (wavefront piatto, gia' normalizzata al
#         flusso totale) -- e' self.image_ref nella tua classe AOS.
#     pwfs_grid : Grid
#         Grid del rivelatore PWFS -- e' self.pwfs_grid nella tua classe AOS.
#     threshold : float
#         Soglia di flusso per considerare un pixel "dentro" la pupilla.
#     smooth_sigma : float
#         Sigma del filtro gaussiano applicato PRIMA della soglia, per
#         eliminare i falsi buchi dovuti al pattern di diffrazione a croce.
#     fill_holes : bool
#         Se True, applica un secondo passaggio di fill_holes come rete
#         di sicurezza.

#     Ritorna
#     -------
#     mask : Field (booleano)
#         Maschera valida, stessa shape di image_ref.
#     """
#     image_shaped = image_ref.shaped  # torna a griglia 2D (da HCIPy Field a array)

#     # 1. Smoothing per eliminare i "baffi" di diffrazione interni
#     image_smooth = ndimage.gaussian_filter(image_shaped, sigma=smooth_sigma)

#     # 2. Sogliatura sull'immagine smussata, non su quella grezza
#     mask = image_smooth > threshold

#     # 3. Fill holes come rete di sicurezza (nel caso rimangano buchi residui)
#     if fill_holes:
#         mask = ndimage.binary_fill_holes(mask)

#     # torna a Field (1D) per essere coerente col resto del codice HCIPy
#     mask_field = hc.Field(mask.ravel().astype(float), pwfs_grid)

#     return mask_field

In [8]:
# mask = build_pwfs_mask(image_ref, pwfs_grid, threshold=2.60112e-5, smooth_sigma=2)

# hc.imshow_field(mask)
# plt.colorbar()
# plt.show()

In [9]:
# def build_geometric_pwfs_mask(aperture_function, pupil_grid_diameter, pwfs_grid):
#     """
#     Costruisce la maschera delle 4 pupille del PWFS per via geometrica,
#     replicando l'aperture originale nei 4 centri noti sul pwfs_grid,
#     invece di stimarla dall'intensita'.

#     Parametri
#     ----------
#     aperture_function : callable
#         La funzione di apertura originale, es. hc.make_magellan_aperture()
#         (la stessa usata per costruire self.magellan_aperture).
#     pupil_grid_diameter : float
#         self.pupil_grid_diameter -- e' anche il valore di 'separation'
#         passato a PyramidWavefrontSensorOptics.
#     pwfs_grid : Grid
#         self.pwfs_grid.

#     Ritorna
#     -------
#     mask : Field (booleano)
#     """
#     separation = pupil_grid_diameter
#     centers = [(+1, +1), (+1, -1), (-1, +1), (-1, -1)]

#     mask = pwfs_grid.zeros()

#     for sx, sy in centers:
#         cx = sx * separation / 2
#         cy = sy * separation / 2
#         # Sposto il grid (non l'apertura) e ci valuto sopra l'apertura originale
#         shifted_grid = pwfs_grid.shifted([-cx, -cy])
#         aperture_i = hc.evaluate_supersampled(aperture_function, shifted_grid, 6)
#         mask = np.maximum(mask, aperture_i)

#     return mask > 0.5

In [10]:
# aperture_function = hc.make_magellan_aperture()  # la stessa usata per self.magellan_aperture
# mask = build_geometric_pwfs_mask(aperture_function, pupil_grid_diameter, pwfs_grid)

# hc.imshow_field(mask)
# plt.colorbar()
# plt.show()

###Shack-Hartmann WFS

### Deformable Mirror and Scientific Camera setup

In [11]:
# num_actuators_across_pupil = 20
# actuator_spacing = telescope_diameter / num_actuators_across_pupil
# print(f'Actuator spacing: {actuator_spacing:.2f} m')
# influence_functions = hc.make_gaussian_influence_functions(pupil_grid, num_actuators_across_pupil, actuator_spacing)
# influence_functions_dense = influence_functions.to_dense()
# print(influence_functions_dense.transformation_matrix.shape)
# illuminated_influence_functions = hc.find_illuminated_actuators(influence_functions_dense, magellan_aperture)
# print(illuminated_influence_functions.transformation_matrix.shape)
# deformable_mirror = hc.DeformableMirror(illuminated_influence_functions)
# num_modes = deformable_mirror.num_actuators
# print(f'Number of actuators: {num_modes}')

In [12]:
# # from hcipy source code modified output
# def illuminated_actuators_mask(basis, aperture, power_cutoff=0.1):
#     '''Find the illuminated modes.

#     A subset of the modes is selected based on the aperture function and a power cutoff.

#     Parameters
#     ----------
#     basis : ModeBasis
#         The mode basis for which we want to find the illuminated modes.
#     aperture : Field or array_like
#         The aperture
#     power_cutoff : scalar
#         The minimal required power over the aperture.

#     Returns
#     -------
#     mask : ndarray
#         The illuminated influence functions mask.
#     '''
#     total_power = np.sum(abs(basis._transformation_matrix)**2, axis=0)
#     masked_power = np.sum(abs(basis._transformation_matrix[aperture > 0])**2, axis=0)
#     _illuminated_actuator_mask = masked_power >= (power_cutoff * total_power)

#     return  _illuminated_actuator_mask

In [13]:
# mask = illuminated_actuators_mask(influence_functions_dense, magellan_aperture)
# print(np.flip(np.array(mask.copy()).reshape(int(np.sqrt(len(mask))), int(np.sqrt(len(mask)))), axis=0))

In [14]:
# # Create the interaction matrix starting from mode basis
# probe_amp = 0.01 * wavelength_wfs # actuator amplitude for probing the response of the system (ex. Boston Micromachines DMs have actuators with a total range of motion between 1.5 and 5.5 nm. https://bostonmicromachines.com/products/deformable-mirrors/standard-deformable-mirrors/)
# slopes = []

# wf = hc.Wavefront(magellan_aperture, wavelength_wfs)
# wf.total_power = 1

# for ind in tqdm(range(num_modes)):
#     if ind % 10 == 0:
#         print("Measure response to mode {:d} / {:d}".format(ind+1, num_modes))
#     slope = 0

#     # Probe the phase response
#     for s in [1, -1]:
#         amp = np.zeros((num_modes,))
#         amp[ind] = s * probe_amp
#         deformable_mirror.actuators = amp

#         dm_wf = deformable_mirror.forward(wf)
#         wfs_wf = pwfs.forward(dm_wf)

#         camera.integrate(wfs_wf, 1)
#         image = camera.read_out()
#         image /= np.sum(image)

#         slope += s * (image-image_ref)/(2 * probe_amp) # finite first derivative approximated with central difference method. Variation in PWFS detector image wrt the activation of a piezo electric actuator is the DM.

#     slopes.append(slope)

# slopes = hc.ModeBasis(slopes) # derivatives of PWFS wrt each DM actuator.

In [15]:
# rcond = 0.08
# reconstruction_matrix = hc.inverse_tikhonov(slopes.transformation_matrix, rcond=rcond, svd=None)

In [16]:
# # Scientific camera: spatial resolution of the system, focal plane image size and resolution
# spatial_resolution = wavelength_wfs / telescope_diameter #Rayleigh criterion for spatial resolution. The smaller the angle the finer the resolution. The larger the telescope diameter, the better. Assumed f=focallenght=1m
# focal_grid = hc.make_focal_grid(q=8, num_airy=20, spatial_resolution=spatial_resolution)
# prop = hc.FraunhoferPropagator(pupil_grid, focal_grid)
# norm = prop(wf).power.max()

#### Secondary alternative, not important:

In [17]:
# # --- Alternative response matrix build using slope estimates (pwfse.estimate) ---
# # This cell constructs a response matrix where each column is the slope response
# # to a small poke on each actuator, using the PyramidWavefrontSensorEstimator.
# pwfse = hc.PyramidWavefrontSensorEstimator(magellan_aperture)
# probe_amp = 0.05 * wavelength_wfs  # small poke for linear response
# response_matrix = []
# wf.total_power = 1
# # compute reference slopes from reference image
# slopes_ref = pwfse.estimate(image_ref)

# for i in tqdm(range(num_modes)):
#     if i % 50 == 0:
#         print(f'Measuring slope response for actuator {i+1} / {num_modes}')
#     slope_acc = 0
#     # Probe both signs and form central difference
#     for s in [1, -1]:
#         deformable_mirror.flatten()
#         amps = np.zeros((num_modes,))
#         amps[i] = s * probe_amp
#         deformable_mirror.actuators = amps

#         dm_wf = deformable_mirror.forward(wf)
#         wfs_wf = pwfs.forward(dm_wf)

#         camera.integrate(wfs_wf, 1)
#         image = camera.read_out()
#         image /= np.sum(image)

#         slopes = pwfse.estimate(image)
#         slope_acc += s * (slopes - slopes_ref) / (2 * probe_amp)

#     response_matrix.append(slope_acc.ravel())

# response_matrix = hc.ModeBasis(response_matrix)

In [18]:
# rcond = 0.05
# reconstruction_matrix = hc.inverse_tikhonov(response_matrix.transformation_matrix, rcond=rcond)
# print('Built reconstruction_matrix from slope-based response (shape):', reconstruction_matrix.shape)

### Atmosphere

In [19]:
# seeing = 1.2 # arcsec @ 500nm (convention)
# outer_scale =  40  # meter  #40
# tau0 = 0.002 # seconds

# fried_parameter = hc.seeing_to_fried_parameter(seeing)
# Cn_squared = hc.Cn_squared_from_fried_parameter(fried_parameter, 500e-9)
# velocity = 0.314 * fried_parameter / tau0

# print('r0   = {0:.1f}cm'.format(fried_parameter * 100))
# print('L0   = {0:.1f}m'.format(outer_scale))
# print('tau0 = {0:.1f}ms'.format(tau0 * 1000))
# print('v    = {0:.1f}m/s'.format(velocity))
# print('Cn^2 = {0:.1f}'.format(Cn_squared))

In [20]:
# layer = hc.InfiniteAtmosphericLayer(pupil_grid, Cn_squared, outer_scale, velocity= velocity)

# phase_screen_phase = layer.phase_for(wavelength_wfs) # in radian
# phase_screen_opd = phase_screen_phase * (wavelength_wfs / (2 * np.pi)) * 1e6  #optical path difference (mn, that's why we multiply by 1e6). Values are relative delays in the wavefront, not absolute phase values.
# hc.imshow_field(phase_screen_opd, cmap='RdBu')
# plt.colorbar()

In [21]:
# plt.figure(figsize=(5, 4))
# anim = hc.FFMpegWriter('atmospheric_turbulence.mp4', framerate=5)

# t_end = 1
# n = 51
# for t in np.linspace(0, t_end, n):
#     layer.t = t

#     phase_screen_phase = layer.phase_for(wavelength_wfs) # in radian
#     phase_screen_opd = phase_screen_phase * (wavelength_wfs / (2 * np.pi)) * 1e6 # in um

#     plt.clf()
#     plt.suptitle('Time %.3f s' % t)

#     plt.subplot(1,1,1)
#     plt.title('Turbulent wavefront [$\\mu$m]')
#     hc.imshow_field(phase_screen_opd, vmin=-6, vmax=6, cmap='RdBu')
#     plt.colorbar()

#     anim.add_frame()

# plt.close()
# anim.close()

# # Show created animation
# anim

In [22]:
# slopes_ref = pwfse.estimate(image_ref)

### Closed loop animation

In [23]:
# from matplotlib import animation, rc
# from IPython.display import HTML
# import warnings
# warnings.filterwarnings("ignore")

In [24]:
# # deformable_mirror_atmos = hc.DeformableMirror(influence_functions)
# # deformable_mirror_atmos.random(0.2 * wavelength_wfs)
# layer.reset()
# deformable_mirror.flatten()
# # gain = 0.2
# # leakage = 0
# gain = 0.5
# leakage = 0.05
# num_iterations = 100
# burn_in_iterations = 5
# delta_t = 1E-3

# # coro = PerfectCoronagraph(VLT_aperture, 4)

# long_exposure = focal_grid.zeros()
# # long_exposure_coro = focal_grid.zeros()

# # Set up animation
# plt.figure(figsize=(12, 8))
# anim = hc.FFMpegWriter('AO_simulation_with_turbulence.mp4', framerate=10)

# strehl_list = []

# for timestep in tqdm(range(num_iterations)):
#     layer.t = timestep * delta_t
#     phase_screen_phase = layer.phase_for(wavelength_wfs) # in radian
#     phase_screen_opd = phase_screen_phase * (wavelength_wfs / (2 * np.pi)) * 1e6 # in um


#     # Propagate through atmosphere and deformable mirror.
#     wf_wfs_after_atmos = layer(wf)
#     # wf_wfs_after_atmos = deformable_mirror_atmos(wf) # the DM is part of the atmosphere, so we propagate through the DM and then through the atmosphere. The DM is not in the control loop, it's just a static phase screen that adds to the atmospheric turbulence.
#     wf_wfs_after_dm = deformable_mirror(wf_wfs_after_atmos)

#     # Propagate through P-WFS
#     wf_wfs_on_pwfs = pwfs(wf_wfs_after_dm) #shwfs(magnifier(wf_wfs_after_dm))

#     # Propagate the NIR wavefront
#     #wf_sci_focal_plane = propagator(deformable_mirror(layer(wf_sci)))
#     #wf_sci_coro = propagator(coro(deformable_mirror(layer(wf_sci))))
#     psf = prop(wf_wfs_after_dm).power

#     # Read out WFS camera
#     camera.integrate(wf_wfs_on_pwfs, delta_t)
#     wfs_image = camera.read_out()
#     #wfs_image = hc.large_poisson(wfs_image).astype('float')
#     wfs_image /= np.sum(wfs_image)

#     # # Accumulate long-exposure image
#     if timestep >= burn_in_iterations:
#         long_exposure += psf / (num_iterations - burn_in_iterations)
#     #     long_exposure_coro += wf_sci_coro.power / (num_iterations - burn_in_iterations)

#     # # Calculate slopes from WFS image
#     # slopes = pwfse.estimate(wfs_image)
#     # slopes -= slopes_ref
#     # slopes = slopes.ravel()
#     diff_image = wfs_image - image_ref

#     # # Perform wavefront control and set DM actuators
#     # deformable_mirror.actuators = (1 - leakage) * deformable_mirror.actuators - gain * reconstruction_matrix.dot(slopes)
#     deformable_mirror.actuators = (1-leakage) * deformable_mirror.actuators - gain * reconstruction_matrix.dot(diff_image) # change actuators position in DM. Classic integrator with leakage(perdita).

#     sr = hc.get_strehl_from_focal(psf, prop(wf).power) #current strehl ratio
#     strehl_list.append(sr)

#     if timestep == num_iterations - 1:
#         print(f'___Strehl Ratio over {num_iterations} iterations___')
#         print(f'Average: {np.mean(strehl_list) * 100:.1f}%')
#         print(f'Std: {np.std(strehl_list) * 100:.1f}%')
#         del strehl_list
#         print(f'LE SR: {hc.get_strehl_from_focal(long_exposure, prop(wf).power) * 100:.1f}%')

#     # Plotting
#     if timestep % 20 == 0:
#         plt.clf()

#         plt.suptitle('Timestep %d / %d' % (timestep, num_iterations))

#         plt.subplot(2,3,1)
#         plt.title('DM surface [nm]') #[$\\mu$m]')
#         hc.imshow_field(magellan_aperture * deformable_mirror.surface * 1e9, cmap='RdBu')#, vmin=-1, vmax=1) #, mask=VLT_aperture)
#         plt.colorbar()

#         plt.subplot(2,3,2)
#         plt.title('WFS image [counts]')
#         hc.imshow_field(wfs_image) #, cmap='inferno')
#         plt.colorbar()

#         plt.subplot(2,3,3)
#         plt.title('Instantaneous PSF ($\\mu$m) [log]')
#         # hc.imshow_field(np.log10(wf_sci_focal_plane.power/ wf_sci_focal_plane.power.max()), vmin=-6, vmax=0, cmap='inferno')
#         hc.imshow_field(np.log10(psf / psf.max() * wf_wfs_after_dm.total_power), cmap='inferno')
#         plt.colorbar()

#         if timestep >= burn_in_iterations:
#             plt.subplot(2,3,4)
#             plt.title('Average PSF at 2.2$\\mu$m [log]')
#             hc.imshow_field(np.log10(long_exposure / long_exposure.max()), vmin=-6, vmax=0, cmap='inferno')
#             plt.colorbar()

#         plt.subplot(2,3,5)
#         plt.title('Turbulent wavefront [$\\mu$m]')
#         hc.imshow_field(magellan_aperture * phase_screen_opd, cmap='RdBu')
#         plt.colorbar()

#         print(f'Timestep {timestep}: Strehl ratio = {sr * 100 :.1f}%') # print strehl ratio

#         anim.add_frame()

# plt.close()
# anim.close()

# # Show created animation
# anim


# More compact code

### Command line arguments

In [25]:
# seeing = 1.2 # arcsec @ 500nm (convention). The smaller the softer the turbulence and easier to correct.
# outer_scale =  40  # larger turbulence diameter, meters
# tau0 = 0.002 # temporal coherence, seconds
# fried_parameter = hc.seeing_to_fried_parameter(seeing) # r0, spatial coherence, meters
# Cn_squared = hc.Cn_squared_from_fried_parameter(fried_parameter, 500e-9)
# velocity = 0.314 * fried_parameter / tau0 # wind velocity

# print('r0   = {0:.1f}cm'.format(fried_parameter * 100))
# print('L0   = {0:.1f}m'.format(outer_scale))
# print('tau0 = {0:.1f}ms'.format(tau0 * 1000))
# print('v    = {0:.1f}m/s'.format(velocity))
# print('Cn^2 = {0:.1f}'.format(Cn_squared))

In [26]:
#previous: 0.7 gain, 0.01 leakage, 0.6 seeing, 0.005 tau vs 0.06, 0.02, 0.5, 0.006

In [74]:
import argparse

def create_args():

    parser = argparse.ArgumentParser(description="Adaptive Optics Closed Loop System Initialization and Simulation Parameters")

    # Adaptive Optics system set up
    parser.add_argument("--wavelength_wfs", type=float, default=842.0E-9, help="Wavelength for the wavefront sensor in meters (default: 842 nm)")
    parser.add_argument("--telescope_diameter", type=float, default=6.5, help="Diameter of the telescope in meters (default: 6.5 m)")
    parser.add_argument("--num_pupil_pixels", type=int, default=70, help="Number of pixels across the pupil grid (default: 70)")###
    parser.add_argument("--num_actuators_across_pupil", type=int, default=4, help="Number of actuators across the pupil (default: 24)") ###
    parser.add_argument("--rcond", type=float, default=0.08, help="Regularization condition number for Tikhonov inversion (default: 0.08)")
    parser.add_argument("--seeing", type=float, default=0.6, help="Seeing in arcseconds at 500nm (default: 1.2 arcsec)") #the more it grows the smaller r_0
    parser.add_argument("--outer_scale", type=float, default=30, help="Outer scale of turbulence in meters (default: 40 m)")###
    parser.add_argument("--tau0", type=float, default=0.005, help="Coherence time of turbulence in seconds (default: 0.002 s)")###
    parser.add_argument("--prob_amp_coeff", type=float, default=0.1, help="Coefficient for actuator amplitude probing (default: 0.01)") ###0.1
    parser.add_argument("--power_cutoff", type=float, default=0.15, help="Power cutoff for illuminated actuators (default: 0.1)")
    parser.add_argument("--modulation_radius", type=int, default=3, help="Radius to modulate modulated pyramid wavefront sensor (default: 3). When 0, WFS is unmodulated")
    parser.add_argument("--modulation_steps", type=int, default=8, help="Iteration steps to modulate pyramid WFS (default: 16)")

    # Calibration of normalisation scales, entropy temperature init
    parser.add_argument("--max_command", type=float, default=2000E-9, help="Maximum command for the actuators (default: 800 nm)") ###
    parser.add_argument("--coeff_max_action", type=float, default=0.3, help="Coefficient multiplying rec rms to set maximum RL action per actuator")
    parser.add_argument("--calib_steps", type=int, default=300, help="Calibration steps used to measure the RMS of reconstruction/commands (default: 300)")
    parser.add_argument("--calib_burn_in", type=int, default=50, help="Calibration burn-in steps used to measure the RMS of reconstruction/commands (default: 50)") #servono?
    parser.add_argument("--calib_n_runs", type=int, default=4, help="Number of runs for calibration (default: 4)")
    parser.add_argument("--alpha_init", type=float, default=0.2, help="Initial SAC temperature alpha (default: 0.2, as in Pou et al.)") # Residual RL correction: start conservatively so SAC cannot undo the integrator.

    # closed loop episode simulation
    parser.add_argument("--gain", type=float, default=0.7, help="Gain for the closed loop control (default: 0.5)")
    parser.add_argument("--leakage", type=float, default=0.01, help="Leakage for the closed loop control (default: 0.05)") ###
    parser.add_argument("--num_iterations", type=int, default=100, help="Number of iterations for the closed loop simulation (default: 100)") #not used
    parser.add_argument("--burn_in_iterations", type=int, default=5, help="Number of burn-in iterations before accumulating long-exposure image (default: 5)")
    parser.add_argument("--delta_t", type=float, default=1E-3, help="Time step for each iteration in seconds (default: 1E-3 s)")

    # SAC parameters
    parser.add_argument("--gamma", type=float, default=0.1, help="Discount factor for SAC (default: 0.1)")
    parser.add_argument("--tau", type=float, default=0.005, help="Soft update coefficient for target networks in SAC (default: 0.005)")
    parser.add_argument("--replay_pool_size", type=int, default=50000, help="Size of the replay pool for SAC (default: 50000, as in Pou et al.)")
    parser.add_argument("--episodes", type=int, default=5, help="Number of TRAIN episodes for SAC (default: 100). Choose an even number to end with an eval episode(default: 100).") # choose an even number to end with an eval episode
    parser.add_argument("--train_each_eval", type=int, default=2, help="N train episodes preceeding each eval episode at current policy (default: 2)")
    parser.add_argument("--episode_steps", type=int, default=10, help="Number of steps per SAC episode (default: 1000, same as the test length)")
    parser.add_argument("--warm_up_steps", type=int, default=0, help="Number of warmup steps for SAC (default: 0)")
    parser.add_argument("--env_steps", type=int, default=1, help="Number of steps in the environment for SAC (default: 1)")
    parser.add_argument("--gradient_steps", type=int, default=1, help="Number of gradient steps for SAC (default: 1)")
    parser.add_argument("--actor_lr", type=float, default=3e-4, help="Learning rate for the actor (default: 1e-4)")
    parser.add_argument("--critic_lr", type=float, default=3e-4, help="Learning rate for the critic (default: 1e-4)")
    parser.add_argument("--alpha_lr", type=float, default=3e-4, help="Learning rate for the temperature parameter alpha (default: 1e-4)")
    parser.add_argument("--batch_size", type=int, default=256, help="Batch size for gradient steps (default: 256)")
    parser.add_argument("--minibatch_size", type=int, default=256, help="Minibatch size for gradient steps (default: 256)")
    parser.add_argument("--target_entropy", type=float, default=-1, help="Target entropy for SAC (default: 0.01)") #valore a caso per ora

    # Performance / logging
    parser.add_argument("--device", type=str, default="auto", help="Device for PyTorch networks: auto, cpu, cuda, cuda:0, ...") #with auto the code decides whether to use either gpu or cpu depending on the device availability
    parser.add_argument("--save_log", action="store_true", help="Log statistics(default: disabled)")  # Instantaneous SR + command statistics remain EVERY AO step. Only LE SR is sampled more sparsely; its final value is always forced. SAC loss/alpha/entropy/std logging remains sparse to avoid CUDA sync overhead.
    parser.add_argument("--log_gradient_every", type=int, default=10, help="Log optimization statistics every N gradient steps (default: 10)")
    parser.add_argument("--le_strehl_every", type=int, default=10, help="Compute/log long-exposure Strehl every N environment steps; the final step is always logged (default: 10)")

    parser.add_argument("--save_video", action="store_true", help="Save an MP4 during final AO test loops (default: disabled)") #store_true (contrario: store_false): quando in linea di comando è aggiunto l'argomento --save_video questa variabile viene considerata True, se non è specificata è False di default.
    parser.add_argument("--video_every", type=int, default=5, help="Record one video frame every N AO steps (default: 5)")
    parser.add_argument("--video_fps", type=int, default=10, help="Output video frame rate (default: 10 fps)")
    parser.add_argument("--video_dpi", type=int, default=100, help="Output video DPI (default: 100)")
    parser.add_argument("--video_path", type=str, default=None, help="Output video path (default: AO_with_RL.mp4 if rl else AO_without_RL.mp4)")

    # Checkpointing / resume
    parser.add_argument("--checkpoint_every", type=int, default=10, help="Save a resumable checkpoint every N completed TRAIN episodes; <=0 disables checkpointing (default: 10)")
    parser.add_argument("--checkpoint_dir", type=str, default="checkpoints", help="Directory containing latest_checkpoint.pth")
    parser.add_argument("--checkpoint_replay_limit", type=int, default=2000, help="Number of most recent replay transitions stored in checkpoints; <=0 stores the full replay buffer (default: 2000)") #TO DO: how to save the whole buffer without having memory issues (Only a part of the most recent transitions are stored here, keeping checkpoint I/O much lighter than serializing the full replay pool)
    parser.add_argument("--resume_checkpoint", type=str, default="", help="Optional checkpoint path from which training is resumed")

    # System running: classical linear integrator or integrator + rl learned policy
    parser.add_argument("--run_classical_reference", action="store_true", help="Run AO system with classical linear integrator as reference (default: True)")
    parser.add_argument("--system_steps", type=int, default=100, help="Number of steps of AO system ran with reinforcement learning (default: 1000)") #and without RL
    parser.add_argument("--num_test_repeats", type=int, default=1, help="Number of tests runned over classic AO system or classic+RL AO system (default: 3)")

    #args, _ = parser.parse_known_args()
    args = parser.parse_args([])

    return args

args = create_args()


### Adaptive Optics System

In [75]:
# a = np.array([1,0,1,1,0,1], dtype=bool)

# print(a)
# matrix = np.array([[2,3,4,5,6,7],[1,2,3,4,5,6],[1,2,3,4,5,6]])
# print(matrix)
# matrix[:, ~a] = 0.0
# print(matrix)

In [76]:
from collections import deque

In [77]:
class AOS(): #adaptive optics system
    """
    Initialize
    Compute construction matrix
    and iterate over time to collect new states
    """


    def __init__(self, args, layer_seed=None):
        # pupil
        self.telescope_diameter = args.telescope_diameter #m
        self.num_pupil_pixels = args.num_pupil_pixels
        self.pupil_grid_diameter = 60/56 * self.telescope_diameter #1.5 * self.telescope_diameter
        self.pupil_grid = hc.make_pupil_grid(self.num_pupil_pixels, self.pupil_grid_diameter) #generates a simple Cartesian Grid with pixel and reference physical dimensions as stated in the parameters.
        self.magellan_aperture = hc.evaluate_supersampled(hc.make_magellan_aperture(), self.pupil_grid, 6) # mak_magellan_aperture() if no arguments telescope diameter is 6.5m. By changing the diameter of the pupil_grid the proportion grid-aperture changes.

        # Point Spread Function
        self.wavelength_wfs = args.wavelength_wfs
        self.wf = hc.Wavefront(self.magellan_aperture, self.wavelength_wfs) #ELECTRIC FIELD (physical, lots of attributes) on a PLANE (distance from source is irrelevant), a photography of its properties.
        self.wf.total_power = 1 # normalization

        # atmospheric layer
        self.seeing = args.seeing # arcsec @ 500nm (convention)
        self.outer_scale = args.outer_scale    # meter  #40
        self.tau0 = args.tau0 # seconds
        self.fried_parameter = hc.seeing_to_fried_parameter(self.seeing)
        print('r0   = {0:.1f}cm'.format(self.fried_parameter * 100))
        self.Cn_squared = hc.Cn_squared_from_fried_parameter(self.fried_parameter, 500e-9)
        self.velocity = 0.314 * self.fried_parameter / self.tau0
        self.layer = hc.InfiniteAtmosphericLayer(self.pupil_grid, self.Cn_squared, self.outer_scale, velocity= self.velocity, seed=layer_seed)
        #subtraction of piston and tip tilt modes from aberration
        self.zbasis = hc.make_zernike_basis(3, self.telescope_diameter, self.pupil_grid)
        self.Z = np.column_stack([np.asarray(z * self.magellan_aperture) for z in self.zbasis])
        self.Z_pinv = np.linalg.pinv(self.Z)

        # deformable mirror
        self.num_actuators_across_pupil = args.num_actuators_across_pupil
        self.actuator_spacing = self.telescope_diameter / self.num_actuators_across_pupil
        print(f'Actuators spacing: {(self.actuator_spacing * 100):.2f} cm')
        self.influence_functions = hc.make_gaussian_influence_functions(self.pupil_grid, self.num_actuators_across_pupil, self.actuator_spacing)
        self.deformable_mirror = hc.DeformableMirror(self.influence_functions)
        self.num_modes = self.deformable_mirror.num_actuators
        self.prob_amp = args.prob_amp_coeff * self.wavelength_wfs # actuator amplitude for probing the response of the system (ex. Boston Micromachines DMs have actuators with a total range of motion between 1.5 and 5.5 nm. https://bostonmicromachines.com/products/deformable-mirrors/standard-deformable-mirrors/)
        # ...
        self.max_command = args.max_command
        self.coeff_max_action = args.coeff_max_action
        self.max_action = None

        # pyramid wavefront sensor
        self.pwfs_grid = hc.make_pupil_grid(140, 2 * self.pupil_grid_diameter)
        self.pwfs = hc.PyramidWavefrontSensorOptics(self.pupil_grid, self.pwfs_grid, separation=self.pupil_grid_diameter, pupil_diameter=self.telescope_diameter, wavelength_0=self.wavelength_wfs, q=3)
        # modulation
        self.modulation_radius = args.modulation_radius                   # radius in λ/D (0 = non modulated)
        self.modulation_steps = args.modulation_steps
        if self.modulation_radius > 0:
          self.pwfs = hc.ModulatedPyramidWavefrontSensorOptics(self.pwfs, modulation=self.modulation_radius * self.wavelength_wfs / self.telescope_diameter, num_steps=self.modulation_steps, fast_modulation_method=True)
        self.camera = hc.NoiselessDetector(self.pwfs_grid)

        self.image_ref = self.acquire_wfs_image(self.wf)
        # self.camera.integrate(self.pwfs.forward(self.wf), 1)
        # self.image_ref = self.read_pwfs(self.wf, )#self.camera.read_out()
        # self.image_ref /= self.image_ref.sum()

        # Scientific camera: spatial resolution of the system, focal plane image size and resolution
        self.spatial_resolution = self.wavelength_wfs / self.telescope_diameter #Rayleigh criterion for spatial resolution. The smaller the angle the finer the resolution. The larger the telescope diameter, the better. Assumed f=focallenght=1m
        self.focal_grid = hc.make_focal_grid(q=8, num_airy=20, spatial_resolution=self.spatial_resolution)
        self.prop = hc.FraunhoferPropagator(self.pupil_grid, self.focal_grid)
        self.reference_psf = self.prop(self.wf).power
        self.norm = self.reference_psf.max()

        # mask
        self.mask = self.illuminated_actuators_mask(args)
        #print(type(self.mask[0]))
        # reconstruction matrix and history
        self.reconstruction_matrix = None
        self.compute_interaction_matrix(args)

        # iteration
        self.max_iterations = args.num_iterations
        self.gain = args.gain
        self.leakage = args.leakage
        self.burn_in_iterations = args.burn_in_iterations
        self.delta_t = args.delta_t
        self.long_exposure = self.focal_grid.zeros()
        self.strehl_list = []

        self.atmos_rms_history = []
        self.residual_rms_history = []

        self.timestep = 0
        self.psf = None

        # # action
        # self._action = np.zeros((self.deformable_mirror.num_actuators)) # action is the command to the DM actuators. It is a vector of length num_actuators_across_pupil^2, which is the number of actuators in the DM. The action is added to the current state of the DM actuators to update their positions.

        # actuators commands
        self.commands = [np.zeros((self.num_modes)), np.zeros((self.num_modes)), np.zeros((self.num_modes))] # a FIFO queue of past 3 actuators commands, most recent ones are added on the right


        # # [v2] normalisation scales for RL state/reward
        self.calib_steps = args.calib_steps
        self.calib_burn_in = args.calib_burn_in
        self.calib_n_runs = args.calib_n_runs
        self.rec_scale = 1
        self.cmd_scale = 1

        # reconstruction
        self.reconstruction = np.zeros((self.num_modes)) # reconstruction of the wavefront from the slopes. Current[0] and future state[1] (= reward) reconstructions.

    # # action setter for shape check (define action as a property and then define a setter for it)
    # #@property
    # def action(self):
    #     return self._action

    # @action.setter
    # def action(self, value):
    #     arr = np.array(value, copy=True)
    #     if arr.shape != (self.deformable_mirror.num_actuators,):
    #         raise ValueError("Invalid action shape, must be the number of actuators: {}".format(self.deformable_mirror.num_actuators))
    #     self._action = arr


    # Propagate through pwfs and integrate wavefronts in output list from sensor forward method
    def acquire_wfs_image(self, wf):
      wavefronts = self.pwfs.forward(wf)

      if isinstance(wavefronts, (list, tuple)):
          for wavefront in wavefronts:
              self.camera.integrate(wavefront, 1.0 / len(wavefronts))
      else:
          self.camera.integrate(wavefronts, 1.0)

      image = self.camera.read_out()
      total = image.sum()

      if total <= 0:
        raise ValueError("WFS image has zero or negative total intensity")

      return image / total

    #utils function for creating interaction matrix between sensor-mirror
    def compute_interaction_matrix(self, args):
        """
        Computes and saves the interaction matrix between the deformable mirror
        and the pyramid wavefront sensor. The interaction matrix is computed by probing
        each actuator of the deformable mirror and measuring the response on the wavefront sensor.
        The resulting slopes are used to compute the reconstruction matrix using Tikhonov
        regularization.
        """

        # Create the interaction matrix starting from mode basis
        slopes = []

        for ind in tqdm(range(self.num_modes)):
            # if ind % 10 == 0:
            #     print("Measure response to mode {:d} / {:d}".format(ind+1, self.num_modes))
            slope = 0

            # Probe the phase response
            for s in [1, -1]:
                amp = np.zeros((self.num_modes,))
                amp[ind] = s * self.prob_amp
                self.deformable_mirror.actuators = amp

                dm_wf = self.deformable_mirror.forward(self.wf)
                image = self.acquire_wfs_image(dm_wf)

                # wfs_wf = self.pwfs.forward(dm_wf)
                # self.camera.integrate(wfs_wf, 1)
                # image = self.camera.read_out()
                # image /= np.sum(image)

                slope += s * (image-self.image_ref)/(2 * self.prob_amp) # finite first derivative approximated with central difference method. Variation in PWFS detector image wrt the activation of a piezo electric actuator is the DM.

            slopes.append(slope)

        slopes = hc.ModeBasis(slopes) # derivatives of PWFS wrt each DM actuator.

        # to have filtered transformation and reconstruction matrix
        # print(slopes.is_sparse, type(slopes.transformation_matrix))
        # mask the transformation matrix with respect to illuminated actuators
        transf_matrix_filtered = slopes.transformation_matrix.copy() #columns correspondent to non-illuminated actuators are set to zero
        transf_matrix_filtered[:, ~self.mask] = 0.0
        self.reconstruction_matrix = hc.inverse_tikhonov(transf_matrix_filtered, rcond=args.rcond, svd=None)
        self.reconstruction_matrix[~self.mask, :] = 0.0

        # # not filtered transformation and reconstruction matrix
        # self.reconstruction_matrix = hc.inverse_tikhonov(slopes.transformation_matrix, rcond=args.rcond, svd=None)
        self.deformable_mirror.flatten()


    # from hcipy source code modified output
    def illuminated_actuators_mask(self, args):
        '''Find the illuminated modes.

        A subset of the modes is selected based on the aperture function and a power cutoff.

        Parameters
        ----------
        basis : ModeBasis
            The mode basis for which we want to find the illuminated modes.
        aperture : Field or array_like
            The aperture
        power_cutoff : scalar
            The minimal required power over the aperture.

        Returns
        -------
        mask : ndarray
            The illuminated influence functions mask.
        '''
        infl_funct_dense = self.influence_functions.to_dense()
        total_power = np.sum(abs(infl_funct_dense._transformation_matrix)**2, axis=0)
        masked_power = np.sum(abs(infl_funct_dense._transformation_matrix[self.magellan_aperture > 0])**2, axis=0)
        _illuminated_actuator_mask = masked_power >= (args.power_cutoff * total_power)

        return  _illuminated_actuator_mask

    def reset(self):
        """
        Reset the adaptive optics system to its initial state
        """
        self.layer.reset(make_independent_realization=True)
        self.deformable_mirror.flatten()
        self.long_exposure = self.focal_grid.zeros()
        self.strehl_list = []

        self.atmos_rms_history = []
        self.residual_rms_history = []

        self.timestep = 0
        self.commands = [np.zeros((self.num_modes)), np.zeros((self.num_modes)), np.zeros((self.num_modes))]
        self._action = np.zeros((self.deformable_mirror.num_actuators))
        self.reconstruction = np.zeros((self.num_modes))
        self.psf = None

        return self

    def compute_reconstruction(self):
        """
        Returns the current wavefront reconstruction from the slopes.
        """
        # update time for the atmospheric layer
        self.layer.t = self.timestep * self.delta_t

        # Propagate through atmosphere and deformable mirror (both changing attributes during iterations)
        #wf_wfs_after_atmos = self.layer(self.wf)
        phase_screen = self.layer.phase_for(self.wavelength_wfs).copy()
        # Remove piston and tip-tilt
        coefficients = self.Z_pinv @ np.asarray(phase_screen)
        phase_screen_filtered = (phase_screen - self.Z @ coefficients)
        # Convert phase to OPD in meters
        opd = phase_screen_filtered * (self.wavelength_wfs / (2 * np.pi))
        # Build the wavefront with filtered turbulence
        wf_wfs_after_atmos = self.wf.copy()
        wf_wfs_after_atmos.electric_field *= np.exp(1j * phase_screen_filtered)

        # wf_wfs_after_atmos.total_power = 1

        wf_wfs_after_dm = self.deformable_mirror(wf_wfs_after_atmos)




        # # =====================================================
        # # DIAGNOSTICS: ATMOSPHERIC AND RESIDUAL RMS
        # # =====================================================

        # pupil_mask = np.asarray(self.magellan_aperture > 0)

        # # 1. Atmospheric OPD after removing piston, tip and tilt
        # opd_atmos = np.asarray(phase_screen_filtered) * (
        #     self.wavelength_wfs / (2 * np.pi)
        # )

        # atmos_rms_nm = np.std(opd_atmos[pupil_mask]) * 1e9

        # # 2. OPD introduced by the deformable mirror
        # # HCIPy reflective DM: optical path difference = 2 * surface
        # dm_opd = 2 * np.asarray(self.deformable_mirror.surface)

        # # 3. Total residual OPD after the DM
        # opd_residual = opd_atmos + dm_opd

        # residual_rms_nm = np.std(opd_residual[pupil_mask]) * 1e9

        # # 4. Store diagnostics
        # self.atmos_rms_history.append(atmos_rms_nm)
        # self.residual_rms_history.append(residual_rms_nm)


        wfs_image = self.acquire_wfs_image(wf_wfs_after_dm)
        # # Propagate through P-WFS
        # wf_wfs_on_pwfs = self.pwfs(wf_wfs_after_dm)
        # # Read out WFS camera
        # self.camera.integrate(wf_wfs_on_pwfs, self.delta_t)
        # wfs_image = self.camera.read_out()
        # wfs_image /= np.sum(wfs_image)

        # Propagate the wavefront to scientific camera
        self.psf = self.prop(wf_wfs_after_dm).power

        # Accumulate long-exposure image
        if self.timestep >= self.burn_in_iterations:
            # self.long_exposure += self.psf / (self.timestep + 1 - self.burn_in_iterations)
            n = self.timestep + 1 - self.burn_in_iterations
            self.long_exposure += (self.psf - self.long_exposure) / n

        # Calculate slopes from WFS image
        diff_image = wfs_image - self.image_ref

        # Perform wavefront control and set DM actuators
        self.reconstruction = (self.reconstruction_matrix.dot(diff_image)) # reconstruction of the wavefront from the slopes.

        self.timestep += 1

    # def __iter__(self):
    #     return self

    # def __next__(self):
    #     """
    #     AOS step function to propagate the wavefront through the atmosphere, deformable mirror,
    #     and pyramid wavefront sensor, and compute the current wavefront reconstruction.
    #     """
    #     if self.max_iterations is not None and self.timestep >= self.max_iterations:
    #         raise StopIteration
    #     self.compute_reconstruction()

    #     self.timestep += 1

    #     return self.reconstruction

    def apply_DM_command(self,action=None):
        """
        Compute and apply command to update the DM actuators based on the current reconstruction and action.
        And statistics of the current state of the system (strehl and long exposure strehl ratios).
        This function should be called after each iteration of the closed loop simulation.
        """
        # change actuators position in DM. Classic integrator with leakage(perdita).
        if action is None:
            action_dm = np.zeros((self.deformable_mirror.num_actuators))
        else:
          # scale action to the maximum allowed value
          action_dm = np.clip(action * self.max_action, -self.max_action, self.max_action)

        # new command
        new_command = (1-self.leakage) * self.deformable_mirror.actuators - self.gain * self.reconstruction + action_dm
        # clip command to the maximum allowed value & mask
        new_command = np.clip(new_command, -self.max_command, self.max_command)
        new_command[~self.mask] = 0  # Mask out the actuators that are not illuminated
        self.deformable_mirror.actuators = new_command
        # otherwise : self.deformable_mirror.actuators = new_command(self, action)

        # DM command statistics (illuminated actuators only)
        dm_commands = np.asarray(self.deformable_mirror.actuators)[self.mask]
        dm_rms_nm = np.sqrt(np.mean(dm_commands**2)) * 1e9
        # dm_max_nm = np.max(np.abs(dm_commands)) * 1e9

        # updating state queue
        if len(self.commands) >= 3:
            self.commands.pop(0)
        self.commands.append(self.deformable_mirror.actuators.copy())

        sr = hc.get_strehl_from_focal(self.psf, self.reference_psf) #current strehl ratio
        self.strehl_list.append(sr)

        # Plotting
        if (self.timestep) % 10 == 0:
            print(f'___Strehl Ratio over {self.timestep} iterations___')
            # print(f'Average: {np.mean(self.strehl_list) * 100:.1f}%')
            # print(f'Std: {np.std(self.strehl_list) * 100:.1f}%')
            print(f'SR: {sr * 100:.1f}%')
            print(f'LE SR: {hc.get_strehl_from_focal(self.long_exposure, self.reference_psf) * 100:.1f}%')
            print(f"DM RMS: {dm_rms_nm:8.1f} nm")
        # if (self.timestep) % 5 == 0:
        #     print(f'___Strehl Ratio of {self.timestep}-th iteration___')
        #     print(f'Average: {np.mean(self.strehl_list) * 100:.1f}%')
        #     print(f'Std: {np.std(self.strehl_list) * 100:.1f}%')
        #     print(f'SR: {sr * 100:.1f}%')
        #     print(f"DM RMS: {dm_rms_nm:8.1f} nm")
        #     print(f"DM MAX: {dm_max_nm:8.1f} nm")
        #     print(f"Atmos RMS: {self.atmos_rms_history[-1]:.1f} nm")
        #     print(f"Residual RMS: {self.residual_rms_history[-1]:.1f} nm")


    # def calibrate_scales(self, n_steps=300, burn_in=50):
    #     """[v2] Run the classical integrator (no RL) and measure the RMS of the reconstruction
    #     and of the DM commands over the illuminated actuators. These scales normalise the RL
    #     state and the reward."""
    #     self.reset()
    #     recs, cmds = [], []
    #     for k in range(n_steps):
    #         self.compute_reconstruction()
    #         self.apply_DM_command()
    #         if k >= burn_in:
    #             recs.append(np.asarray(self.reconstruction)[self.mask].copy())
    #             cmds.append(np.asarray(self.deformable_mirror.actuators)[self.mask].copy())
    #     self.rec_scale = float(np.sqrt(np.mean(np.concatenate(recs) ** 2)))
    #     self.cmd_scale = float(np.sqrt(np.mean(np.concatenate(cmds) ** 2)))
    #     self.reset()
    #     print(f"[calibration] rec RMS = {self.rec_scale*1e9:.2f} nm | command RMS = {self.cmd_scale*1e9:.2f} nm | "
    #           f"max_action / rec RMS = {self.max_action/self.rec_scale:.2f}")


    def calibrate_scales(self):
        """ Run the classical integrator (no RL) and measure the RMS of the reconstruction
          and of the DM commands over the illuminated actuators, over n_steps steps and n_runs runs.
          These scales normalise the RL state and the reward."""
        rec_sumsq = 0.0
        cmd_sumsq = 0.0
        count = 0

        print("\n" + "=" * 72)
        print("STARTING RMS CALIBRATION")
        print("=" * 72)
        for run in range(self.calib_n_runs):
            # Necessario che ogni run utilizzi una
            # realizzazione diversa della turbolenza.
            self.reset()

            for k in range(self.calib_steps):
                self.compute_reconstruction()
                self.apply_DM_command()

                if k >= self.calib_burn_in:
                    rec = np.asarray(self.reconstruction)[self.mask]
                    cmd = np.asarray(self.deformable_mirror.actuators)[self.mask]

                    rec_sumsq += np.sum(rec**2)
                    cmd_sumsq += np.sum(cmd**2)
                    count += rec.size

        self.rec_scale = np.sqrt(rec_sumsq / count)
        self.cmd_scale = np.sqrt(cmd_sumsq / count)
        self.max_action = self.coeff_max_action * self.rec_scale

        self.reset()
        print(f"[calibration] rec RMS = {self.rec_scale*1e9:.2f} nm | command RMS = {self.cmd_scale*1e9:.2f} nm | max action = {self.max_action*1e9:.2f} nm")



    def get_long_exposure_strehl(self):
        """Return the current long-exposure Strehl ratio.

        The diffraction-limited reference PSF is cached at initialization, so
        this method does not perform an additional Fraunhofer propagation.
        """
        if self.timestep <= self.burn_in_iterations:
            return np.nan
        return float(hc.get_strehl_from_focal(self.long_exposure, self.reference_psf))

    # def closed_loop_step_anim():
    #     pass


In [78]:
# env = AOS(args)
# env.compute_interaction_matrix(args)
# env_ = iter(env)
# for timestep in range(3):
#     next(env_)
#     state = np.array(env.state[0])
#     print(state[:5])  # Print the first 5 actuator commands of the current state

### Actor and Critic networks

In [79]:
# Actor: policy function
import torch
import torch.nn as nn
from torch import Tensor, cuda
from typing import Optional, Tuple
from torch.distributions import Normal

The actor parametrizes the policy function. It takes the state as input and outputs a multivariated normal distribution (of independent variables) over the action space, from which the final action is then sampled.
The critic ... .

In [80]:
@torch.no_grad()  # of course we don't want the initialization to be tracked by the computational graph.
def weights_init_(m: nn.Module) -> None:
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        #nn.init.kaiming_normal_(m.weight, nonlinearity="relu") # better for networks using ReLu activation functions (https://www.geeksforgeeks.org/python/select-the-right-weight-for-deep-neural-network-in-pytorch/)
        nn.init.xavier_normal_(m.weight)
        if m.bias is not None:
            nn.init.zeros_(m.bias)

In [81]:
# Actor: policy function
class Actor(nn.Module):
    """
    Simple spatial actor taking the state as input and returning the (!!!) normalized action (components in [-1,1]) and its log probability.
    """

    def __init__(self, in_channels: int = 4, hidden_dim: int = 64, num_layers: int = 2, kernel_size: int = 3, eps: float = 1e-8, probe_amp: float = 5e-9) -> None:
        """
        Parameters:
        -----------
        in_channels: int
            Number of input channels (default: 4).
        hidden_dim: int
            Number of hidden channels (default: 64).
        num_layers: int
            Number of hidden convolutional layers apart from means and std level (default: 2).
        kernel_size: int
            Size of the convolutional kernels (default: 3x3).
        eps: float
            Small value to prevent numerical issues (default: 1e-8).
        probe_amp: float
            Actuators amplitude probe (default: 5e-9).
        """
        super().__init__()
        padding = 1
        self.eps = eps #again useful for numerical stability
        self.probe_amp = probe_amp # amplitude of the probe

        # activation function
        self.act = nn.ReLU()

        # layers
        self.layers = nn.ModuleList()

        #first hidden layer
        self.layers.append(nn.Conv2d(in_channels, hidden_dim, kernel_size=kernel_size, padding=padding))
        #2nd to nth hidden layers
        for _ in range(num_layers -1 ):
            self.layers.append(nn.Conv2d(hidden_dim, hidden_dim, kernel_size=kernel_size, padding=padding))
        #last level layers
        self.mean_lay = nn.Conv2d(hidden_dim, 1, kernel_size=kernel_size, padding=padding)
        self.log_std_lay = nn.Conv2d(hidden_dim, 1, kernel_size=kernel_size, padding=padding)


        #initialize weights
        self.apply(weights_init_)
        nn.init.zeros_(self.mean_lay.weight)
        nn.init.zeros_(self.log_std_lay.weight)
        # Keep the residual policy close to zero at the start of training.
        # exp(-3) ~= 0.05, so exploration is initially much smaller than max_action.
        nn.init.constant_(self.log_std_lay.bias, -3.0)

        self.mean = None
        self.log_std = None


    def forward(self, x: Tensor) -> Tensor:
        """
        Input: (Bsz, 4channels, sqrt(num actuators), sqrt(num actuators)) state tensor.
        Output: (Bsz, 1, sqrt(num actuators), sqrt(num actuators)) mean and log_std tensors.
        """
        for conv in self.layers:
            x = conv(x)

            x = self.act(x)

        #output
        self.mean = self.mean_lay(x)
        self.log_std = self.log_std_lay(x)
        self.log_std = torch.clamp(self.log_std, -20, 2) # clamp log_std to prevent numerical issues with very small or large standard deviations

        return self.mean, self.log_std


    def sample(self, state) -> Tensor:
        """
        Sample an action from the policy distribution and scale it to the range of DM actuator influence.

        Output:
        ---------
        action :
            (Bsz, 1, sqrt(num actuators), sqrt(num actuators)) actions tensor (for each actuator)
        log_prob :
            (Bsz, 1, sqrt(num actuators), sqrt(num actuators)) log probabilities tensor (for each actuator)

        """
        #forward pass
        mean, log_std = self.forward(state)

        # back to std
        std = log_std.exp()

        #sampling of a 2D map action for each batch
        normal = Normal(mean, std)
        sample = normal.rsample()

        # clamp action to [-1,1]
        action = torch.tanh(sample) # scale action to the range of DM actuator influence (nm). The output of the network is in the range [-probe_amp, probe_amp], which is the range of motion for the DM actuators. This ensures that the actions taken by the policy are within the physical limits of the DM.
        #  # 1. Stampa le dimensioni (shape) dello strato intermedio
        # print("Shape del Feature Layer:", action.shape)
        # # 2. Stampa i valori numerici del tensore
        # print("Valori del Feature Layer:\n", action)

        # independent action components log prob
        log_prob = normal.log_prob(sample)
        # change of variable of a probability density function
        log_prob -= torch.log(1 - action.pow(2).clamp(min=0, max=1) + self.eps) #action.pow(2) could be slightly larger than 1 due to numerical issues, so we clamp it to 1. eps is added to prevent log(0)
        # # log prob of action (sum of log probabilities of independent components)
        # log_prob = log_prob.sum((2,3), keepdim=False)  # reduce tensor to (Bsz, 1) to have 1 log_prob scalar value per batch.

        # clamp action to the physical range of DM actuators (nm) (rescaling postponed to the time of actual application of the command to the actuators)
        # prob_amp = args.prob_amp_coeff * args.wavelength_wfs # scale action to the range of DM actuator influence (nm). The output of the network is in the range [-probe_amp, probe_amp], which is the range of motion for the DM actuators. This ensures that the actions taken by the policy are within the physical limits of the DM.
        # action = action * args.max_action

        #-------------------------
        # STILL MISSING:
        # actuators masking
        #-------------------------

        return action, log_prob

    def deterministic_action(self, state) -> Tensor:
        """Return the deterministic SAC action used for evaluation/test.

        The deterministic policy uses tanh(mean(s)); no Gaussian sample is drawn.
        Training still uses ``sample()`` and therefore remains stochastic.
        """
        mean, _ = self.forward(state)
        return torch.tanh(mean)



In [82]:
# Critic: state value function Q
class Critic(nn.Module):
    """
    Simple critic taking the state and action as input and returning the state-action value as output.
    """

    def __init__(self, in_channels: int = 4, action_channels: int = 1, hidden_dim: int = 64, num_layers: int = 2, kernel_size: int = 3) -> None:
        """
        Parameters:
        -----------
        in_channels: int
            Number of input channels (default: 4).
        action_channels: int
            Number of action channels (default: 1).
        hidden_dim: int
            Number of hidden channels (default: 64).
        num_layers: int
            Number of hidden convolutional layers (default: 3).
        kernel_size: int
            Size of the convolutional kernels (default: 3x3).
        """
        super().__init__()
        padding = 1

        # activation function
        self.act = nn.ReLU()

        # layers
        self.layers = nn.ModuleList()
        self.layers2 = nn.ModuleList()

        #first hidden layer
        self.layers.append(nn.Conv2d(in_channels + action_channels, hidden_dim, kernel_size=kernel_size, padding=padding))
        self.layers2.append(nn.Conv2d(in_channels + action_channels, hidden_dim, kernel_size=kernel_size, padding=padding))
        #2nd to last hidden layers
        for _ in range(num_layers-1):
            self.layers.append(nn.Conv2d(hidden_dim, hidden_dim, kernel_size=kernel_size, padding=padding))
            self.layers2.append(nn.Conv2d(hidden_dim, hidden_dim, kernel_size=kernel_size, padding=padding))
        # output layer
        self.layers.append(nn.Conv2d(hidden_dim, 1, kernel_size=kernel_size, padding=padding))
        self.layers2.append(nn.Conv2d(hidden_dim, 1, kernel_size=kernel_size, padding=padding))

        self.apply(weights_init_)


    def forward(self, state: Tensor, action: Tensor) -> Tensor:
        """
        Inputs:
        ----------
        state: tensor
            (Bsz, 4channels, sqrt(num actuators), sqrt(num actuators))
        action: tensor
            (Bsz, 4channels, sqrt(num actuators), sqrt(num actuators))

        Outputs:
        ----------
        q1: tensor
            (Bsz, 1, sqrt(num actuators), sqrt(num actuators)) state-action value for the first critic
        q2: tensor
            (Bsz, 1, sqrt(num actuators), sqrt(num actuators)) state-action value for the second critic
        """
        x = torch.cat([state, action], dim=1)
        q1 = x.clone()
        q2 = x.clone()

        # first critic
        for layer in self.layers[:-1]:
            q1 = self.act(layer(q1))
        q1 = self.layers[-1](q1)

        # second critic
        for layer in self.layers2[:-1]:
            q2 = self.act(layer(q2))
        q2 = self.layers2[-1](q2)

        return q1, q2 #2 critic output, the min will be taken to avoid policy positive bias, i.e. to overestimate policy value.


To build the target critic: we generate the critic instance and the target critic instance, and consequently we copy the former weights to the latter. The target critic is then updated with a soft update, which is a weighted average of the target critic and the critic weights.

In [83]:
# target critic (output two Q, then to be compared to extract the min)

### Replay pool

In [84]:
import itertools as it

class ReplayPool:
    """Fixed-capacity circular replay buffer with O(1) insertion."""
    def __init__(self, max_size: int = int(1e6)) -> None:
        self.max_size = int(max_size)
        self.storage = []
        self.next_idx = 0

    def __len__(self):
        return len(self.storage)

    def append(self, item):
        if not isinstance(item, tuple) or len(item) != 4:
            raise ValueError("Item must be a tuple of (state, action, reward, next_state)")

        if len(self.storage) < self.max_size:
            self.storage.append(item)
        else:
            self.storage[self.next_idx] = item

        self.next_idx = (self.next_idx + 1) % self.max_size

    def sample_batch(self, batch_size: int):
        if batch_size > len(self.storage):
            raise ValueError("batch_size larger than replay size")

        return np.random.choice(len(self.storage), size=batch_size, replace=False)

    def format_batch(self, indexes):
        batch = [self.storage[int(i)] for i in indexes]
        states, actions, rewards, next_states = zip(*batch)

        return (
            np.stack(states),
            np.stack(actions),
            np.stack(rewards),
            np.stack(next_states),
        )

    def sample_batch_iterator(self, batch_size: int, minibatch_size: int, drop_last: bool = False):
        indexes = self.sample_batch(batch_size)

        for i in range(0, len(indexes), minibatch_size):
            minibatch_indexes = indexes[i:i + minibatch_size]
            if drop_last and len(minibatch_indexes) < minibatch_size:
                continue
            yield self.format_batch(minibatch_indexes)

    def pop(self):
        if not self.storage:
            raise IndexError("pop from empty replay pool")
        # Manual debug utility only; not part of normal FIFO semantics.
        item = self.storage.pop()
        self.next_idx = len(self.storage) % self.max_size
        return item

    def _chronological_storage(self):
        """Return transitions ordered from oldest to newest."""
        if len(self.storage) < self.max_size or self.next_idx == 0:
            return list(self.storage)
        return list(self.storage[self.next_idx:]) + list(self.storage[:self.next_idx])

    def state_dict(self, max_items=None):
        """Serializable replay snapshot.

        ``max_items`` bounds checkpoint size. A positive value keeps only
        the most recent transitions; None or <=0 stores the full buffer.
        """
        ordered = self._chronological_storage()
        if max_items is not None and int(max_items) > 0:
            ordered = ordered[-int(max_items):]
        return {
            "max_size": self.max_size,
            "storage": ordered,
            "next_idx": len(ordered) % self.max_size,
        }

    def load_state_dict(self, state):
        """Restore a replay snapshot."""
        self.max_size = int(state.get("max_size", self.max_size))
        self.storage = list(state.get("storage", []))
        if len(self.storage) > self.max_size:
            self.storage = self.storage[-self.max_size:]
        self.next_idx = int(state.get("next_idx", len(self.storage) % self.max_size))
        if len(self.storage) < self.max_size:
            self.next_idx = len(self.storage) % self.max_size


In [85]:
# import itertools as it

# a = range(9)
# print(a)

# for i in it.batched(a, 3):
#     print(i)

In [86]:
# # init environment and start iterator
# env = AOS(args)

In [87]:
# print(env.deformable_mirror.actuators.shape)
# array = [i for i in range(9)]
# array = np.array(array)
# print(array)
# sqrt = np.sqrt(array.size)
# matrix = array.reshape(int(sqrt), int(sqrt))
# matrix = np.flip(matrix, axis=0)
# print(matrix)
# print(matrix.shape)


In [88]:
# #init actor
# actor = Actor()

# # init critic
# critic = Critic()

# # init target critic
# target_critic = Critic()
# target_critic.load_state_dict(critic.state_dict()) # copy weights from critic to target critic

# # init alpha, temperature coefficient for entropy regularization in SAC.
# alpha = torch.tensor(0.0, requires_grad=True)

# #init replay pool
# replay = ReplayPool(max_size=100)

In [89]:
# # environment loop
# warm_up_steps = 6
# iterations = 50

# pending_state = None
# pending_action = None

# for timestep in range(iterations):
#     # not saving in replay pool
#     if timestep < warm_up_steps:
#         env.compute_reconstruction() # compute and append current reconstruction to reconstruction deque
#         env.apply_DM_command() # compute and append current command to commands deque
#         continue

#     # saving in replay pool
#     # reconstruction_t
#     env.compute_reconstruction()
#     rec = np.flip(np.array(env.reconstruction.copy()).reshape(int(np.sqrt(len(env.reconstruction))), int(np.sqrt(len(env.reconstruction)))), axis=0)


#     # state_t
#     commands = [np.flip(np.array(i).reshape(int(np.sqrt(len(i))), int(np.sqrt(len(i)))), axis=0) for i in env.commands]  # Reshape each command to (num_modes, num_modes)
#     state = np.concatenate((np.array(commands), rec[np.newaxis, :, :]), axis=0)  # Add current reconstruction to the commands array

#     # print state shape before converting to tensor
#     # print(f"State shape before tensor conversion: {state.shape}")  # Print the shape of the state array
#     state = torch.tensor(state, dtype=torch.float32).unsqueeze(0) # Add batch and channel dimensions
#     # print(f"State shape: {state.shape}")  # Print the shape of the state tensor


#     # action_t
#     actor.eval()
#     act, _ = actor.sample(state)
#     act = act.detach().numpy().squeeze()
#     #print(f"Action shape: {act.shape}")  # Print the shape of the action tensor
#     env.action = act
#     # command_t
#     env.apply_DM_command()

#     # replay pool tuple for previous timestep (t-1)
#     if pending_state is not None and pending_action is not None:
#         replay.append((pending_state, pending_action, rec, state))

#     # pending state_t and action_t
#     pending_state = state
#     pending_action = act

# #print(f"Replay pool size: {len(replay)}")  # Print the size of the replay pool

### (Dataset)

In [90]:
# def minibatch_to_torch(states, actions, rewards, next_states):
#     """
#     Turns numpy arrays into torch tensors and reshapes them to the correct dimensions for the SAC algorithm
#     """
#     # print(f"States shape: {states.shape}")  # Print the shape of the states array
#     # print(f"Actions shape: {actions.shape}")  # Print the shape of the actions array
#     # print(f"Rewards shape: {rewards.shape}")  # Print the shape of the rewards array
#     # print(f"Next states shape: {next_states.shape}")  # Print the shape of the next states array
#     # print(f"First element of rewards shape: {rewards[0].shape}")  # Print the shape of the first element of the rewards array
#     # print(f"first element of action shape: {actions[0].shape}")  # Print the shape of the first element of the actions array

#     # by default tensors created from numpy arrays have requires_grad=False
#     next_states = torch.as_tensor(next_states).squeeze()
#     actions = torch.as_tensor(actions).unsqueeze(1)
#     states = torch.as_tensor(states).squeeze()
#     rewards = torch.as_tensor(rewards).unsqueeze(1)

#     return states, actions, rewards, next_states


In [91]:
# next_states = torch.as_tensor(next_states).squeeze()
# with torch.no_grad():
#     next_action, next_log_pi = actor.sample(next_states)

# print(next_action.shape)
# print(next_log_pi.shape)

In [92]:
# actions = torch.as_tensor(actions).unsqueeze(1)
# states = torch.as_tensor(states).squeeze()

# print(actions.shape, states.shape)

In [93]:
# qvalues = torch.min(critic(states, actions))
# print(qvalues.shape)

In [94]:
# qvalues_target = torch.min(target_critic(next_states, next_action))
# print(qvalues_target.shape)

### (Update Losses)

In [95]:
# import torch.optim as optim
# actor_optim = optim.Adam(actor.parameters())
# critic_optim = optim.Adam(critic.parameters())
# alpha_optim = optim.Adam([alpha])

In [96]:
"""
Update section

1. from D collect a batch of data

2. critic update
    batch:
        state, actions from D to compute qvalue,
        reward from D
        next state from D
        next action and next log_pi from current actor evaluated in next state, for qvalue_target and log_pi
        qvalue from critic evaluated in action and state
        qvalue_target from target critic evaluated in next action and next state
    compute loss with targetQ and pi in eval mode and Q in train mode
    set the optimizer which will perform the gradient descent following a specific algorithm
    go backward on the loss and take a gradient descent step

3. actor update (consider alpha fixed)
    batch:
        state from D
        action and log_pi from actor in train mode (???)
        qvalue from critic with parameters action and state
    compute loss with Q in eval mode and pi in train mode
    optimizer
    bacward
    step

4. alpha update
    batch:
        current alpha
        log_pi from actor
        target entropy
    set target entropy somewhere!
    compute loss
    optimizer
    backward
    step

5. target Q update
    compute weighted mean of target and Q weights
"""

'\nUpdate section\n\n1. from D collect a batch of data\n\n2. critic update\n    batch:\n        state, actions from D to compute qvalue,\n        reward from D\n        next state from D\n        next action and next log_pi from current actor evaluated in next state, for qvalue_target and log_pi\n        qvalue from critic evaluated in action and state\n        qvalue_target from target critic evaluated in next action and next state\n    compute loss with targetQ and pi in eval mode and Q in train mode\n    set the optimizer which will perform the gradient descent following a specific algorithm\n    go backward on the loss and take a gradient descent step\n\n3. actor update (consider alpha fixed)\n    batch:\n        state from D\n        action and log_pi from actor in train mode (???)\n        qvalue from critic with parameters action and state\n    compute loss with Q in eval mode and pi in train mode\n    optimizer\n    bacward\n    step\n\n4. alpha update\n    batch:\n        curr

In [97]:
# # critic loss
# def compute_critic_loss(qvalues, qvalues_target, rewards, alpha, next_log_pi, gamma):
#     loss = 0.5 * (qvalues - (rewards + torch.mul(gamma, qvalues_target) - torch.mul(alpha, next_log_pi))).pow(2)
#     loss = loss.mean()

#     return loss

# # update critic
# def update_critic(
#         states, actions, rewards, next_states, alpha, gamma=args.gamma
#         ):

#     alpha = alpha.detach()
#     with torch.no_grad():
#         next_actions, next_log_pi = actor.sample(next_states)
#         qvalues1_target, qvalues2_target = target_critic(next_states, next_actions)
#         qvalues_target = torch.min(qvalues1_target, qvalues2_target)

#     qvalues1, qvalues2 = critic(states, actions)
#     qvalues = torch.min(qvalues1, qvalues2)

#     # print(f"next_action {next_actions.shape}, \n next_log_pi {next_log_pi.shape}, \n qvalues1_target {qvalues1_target.shape}, \n qvalues2_target {qvalues2_target.shape}, \n qvalues_target {qvalues_target.shape}, \n qvalues1 {qvalues1.shape}, \n qvalues2 {qvalues2.shape}, \n qvalues {qvalues.shape}")

#     critic_loss = compute_critic_loss(qvalues, qvalues_target, rewards, alpha, next_log_pi, gamma)

#     critic_optim.zero_grad()
#     critic_loss.backward()
#     critic_optim.step()

In [98]:
# # actor loss
# def compute_actor_loss(log_pis, qvalues, alpha):
#     loss = (torch.mul(alpha,log_pis) - qvalues) # log_pi is the log probability of the action taken by the actor, qvalues is the state-action value from the critic. The loss is the squared difference between these two quantities, which encourages the actor to take actions that maximize the expected return as estimated by the critic.
#     loss = loss.mean()

#     return loss

# # update actor (separate function from actor loss for modularity reasons)
# def update_actor(
#         states, alpha
#         ):

#     alpha = alpha.detach()
#     actions, log_pis = actor.sample(states)
#     with torch.no_grad():
#         qvalues1, qvalues2 = critic(states, actions)
#         qvalues = torch.min(qvalues1, qvalues2)

#     actor_loss = compute_actor_loss(log_pis, qvalues, alpha)

#     actor_optim.zero_grad()
#     actor_loss.backward()
#     actor_optim.step()

#     return log_pis

In [99]:
# # alpha loss
# def compute_alpha_loss(log_pis, alpha, target_entropy):
#     loss = -torch.mul(alpha, (log_pis + target_entropy).detach()) # detach() is used to prevent gradients from flowing through the log_pi term, which is not needed for the alpha update. The loss is the negative of the product of alpha and the difference between the log probability of the action and the target entropy. This encourages alpha to increase when the policy is too deterministic (log_pi < -target_entropy) and decrease when the policy is too stochastic (log_pi > -target_entropy).
#     loss = loss.mean()

#     return loss

# # update alpha
# def update_alpha(
#         log_pis, target_entropy=-1.0
#         ):
#     log_pis = log_pis.detach()

#     alpha_loss = compute_alpha_loss(log_pis, alpha, target_entropy)

#     alpha_optim.zero_grad()
#     alpha_loss.backward()
#     alpha_optim.step()

In [100]:
# # update target Q state action function
# def update_target_critic(tau=args.tau):
#     for target_param, param in zip(target_critic.parameters(), critic.parameters()):
#         target_param.data.copy_(tau * param.data + (1 - tau) * target_param.data)

In [101]:
# print(alpha.shape)

In [102]:
# # gradient steps
# for minibatch in replay.sample_batch_iterator(batch_size=40, minibatch_size=10):
#     states, actions, rewards, next_states = minibatch_to_torch(*minibatch)
#     # print(states.shape)
#     # print(actions.shape)
#     # print(rewards.shape)
#     # print(next_states.shape)

#     # update critic
#     update_critic(states, actions, rewards, next_states, alpha)

#     # update actor
#     log_pis = update_actor(states, alpha)

#     # update alpha
#     update_alpha(states, log_pis)

#     # update target critic
#     update_target_critic()

#     # we could save loss statistics for actor and critic especially.

### Utility functions SAC class

In [103]:
import torch.optim as optim

def resolve_device(device_arg="auto"):
    if device_arg == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(device_arg)

In [104]:
def initialise(args):
    """Initialising  """
    # HCIPy environment remains CPU/NumPy based.
    env = AOS(args)

    device = resolve_device(getattr(args, "device", "auto"))

    actor = Actor().to(device)
    critic = Critic().to(device)
    target_critic = Critic().to(device)
    target_critic.load_state_dict(critic.state_dict())
    target_critic.requires_grad_(False)

    # Temperature parameter must live on the same device as the networks.
    # [v2] alpha_init = 0.2 (Pou et al.) instead of alpha = 1
    log_alpha = torch.full((), float(np.log(args.alpha_init)), device=device, requires_grad=True)
    #log_alpha = torch.full((), float(np.log(getattr(args, "alpha_init", 0.2))), device=device, requires_grad=True)

    replay = ReplayPool(max_size=args.replay_pool_size)

    actor_optim = optim.Adam(actor.parameters(), lr=args.actor_lr)
    critic_optim = optim.Adam(critic.parameters(), lr=args.critic_lr)
    alpha_optim = optim.Adam([log_alpha], lr=args.alpha_lr)

    # Fixed convolution shapes benefit from cuDNN autotuning.
    if device.type == "cuda":
        torch.backends.cudnn.benchmark = True
        # Faster float32 matrix/convolution kernels on Ampere+ while keeping float32 API.
        torch.backends.cuda.matmul.allow_tf32 = True
        torch.backends.cudnn.allow_tf32 = True

    return env, actor, critic, target_critic, log_alpha, replay, actor_optim, critic_optim, alpha_optim, device


In [105]:
def minibatch_to_torch(states, actions, rewards, next_states, device):
    """Convert a sampled NumPy minibatch and move it to the selected torch device.

    Replay data stay on CPU; only sampled minibatches are transferred to CUDA.
    Shapes are kept explicit (no generic squeeze), which also makes batch_size=1 safe.
    """
    states = torch.as_tensor(states, dtype=torch.float32, device=device)
    next_states = torch.as_tensor(next_states, dtype=torch.float32, device=device)
    actions = torch.as_tensor(actions, dtype=torch.float32, device=device).unsqueeze(1)
    rewards = torch.as_tensor(rewards, dtype=torch.float32, device=device).unsqueeze(1)

    return states, actions, rewards, next_states


In [106]:
def rl_control_step(env, actor, device=None, deterministic=False):
    """Run one RL control step.

    HCIPy stays on CPU. Only the compact state is copied to the actor device; the
    action is copied back once because the physical/HCIPy DM update is CPU based.
    ``deterministic=True`` uses tanh(mean(s)); training keeps Gaussian sampling.
    Replay data are returned as NumPy arrays so the replay buffer does not occupy GPU RAM.
    """
    if device is None:
        device = next(actor.parameters()).device

    side = int(np.sqrt(env.num_modes))

    rec = np.flip(np.asarray(env.reconstruction).reshape(side, side), axis=0).copy()
    # --------------------------------------------------------------------------
    # Commands and reconstruction are both optical-path commands in metres.
    # Normalising them on the same scale avoids an almost invisible residual
    # channel and gives the critic an O(1) reward.
    # [v2] normalise with the empirically calibrated RMS (env.rec_scale / env.cmd_scale) and
    # mask the non-illuminated actuators. The reward is per actuator (no division by num_modes:
    # the losses already average over the valid actuators).
    # --------------------------------------------------------------------------
    mask2d = np.flip(np.asarray(env.mask, dtype=np.float32).reshape(side, side), axis=0)
    # rec = (rec * mask2d).astype(np.float32, copy=False)
    rec_norm = ((rec / env.rec_scale) * mask2d).astype(np.float32, copy=False)
    rew = -np.square(rec_norm)

    commands = np.stack([np.flip(np.asarray(command).reshape(side, side), axis=0)
        for command in env.commands]).astype(np.float32, copy=False)
    commands /= env.cmd_scale

    state_np = np.concatenate((commands, rec_norm[np.newaxis, :, :]), axis=0).astype(np.float32, copy=False)
    state_t = torch.from_numpy(np.ascontiguousarray(state_np)).unsqueeze(0).to(device)

    with torch.inference_mode():
        if deterministic:
            action_t = actor.deterministic_action(state_t)
        else:
            action_t, _ = actor.sample(state_t)

    # One GPU->CPU transfer is unavoidable while HCIPy/DM remains NumPy based.
    action_np = action_t.squeeze(0).squeeze(0).cpu().numpy()
    action_np = action_np * mask2d  # [v2] zero action on non-illuminated actuators (also in the replay)
    action_flat = np.flip(action_np, axis=0).reshape(-1)

    env.apply_DM_command(action=action_flat)

    # Replay uses (C,H,W), (H,W), (H,W), (C,H,W) NumPy objects.
    return rew.astype(np.float32, copy=False), state_np, action_np.astype(np.float32, copy=False)


#### RLLogger Class

In [107]:
import glob
import os
import shutil
import pandas as pd


class RLLogger:
    """Low-overhead logger with rare incremental persistence.

    - Instantaneous Strehl and DM command statistics are logged every environment step.
    - Optimization statistics can be downsampled with ``gradient_log_every``.
    - Long-exposure Strehl can be downsampled with ``le_strehl_every``.
    - During long training runs, buffered records are flushed only at rare checkpoint
      boundaries into small CSV shards. This keeps RAM/checkpoints bounded without
      losing the per-step Strehl/command history.
    """

    COLUMNS = [
        "phase", "type",
        "episode", "episode_step", "global_env_step", "gradient_step",
        "strehl", "le_strehl",
        "critic_loss", "actor_loss", "alpha_loss", "alpha", "entropy", "policy_std",
        "command_min", "command_q25", "command_median", "command_q75",
        "command_max", "command_abs_max", "command_rms",
        "saturation_fraction", "delta_command_rms",
    ]

    def __init__(self, gradient_log_every=10, le_strehl_every=10):
        self.records = []
        self.gradient_log_every = max(1, int(gradient_log_every))
        self.le_strehl_every = int(le_strehl_every)

    @staticmethod
    def _command_statistics(command, previous_command, saturation_limit):
        cmd_full = np.asarray(command)
        valid = np.isfinite(cmd_full)
        cmd = cmd_full[valid]

        if cmd.size == 0:
            return [np.nan] * 9

        q25, median, q75 = np.quantile(cmd, [0.25, 0.50, 0.75])

        delta_rms = np.nan
        if previous_command is not None:
            prev_full = np.asarray(previous_command)
            prev = prev_full[valid]
            delta_rms = float(np.sqrt(np.mean(np.square(cmd - prev))))

        return [
            float(cmd.min()),
            float(q25),
            float(median),
            float(q75),
            float(cmd.max()),
            float(np.max(np.abs(cmd))),
            float(np.sqrt(np.mean(np.square(cmd)))),
            float(np.mean(np.abs(cmd) >= saturation_limit)),
            delta_rms,
        ]

    def log_environment_step(
        self, *,
        phase,
        episode,
        episode_step,
        global_env_step,
        strehl,
        command,
        max_command,
        previous_command,
        le_strehl=np.nan,
    ):
        """Log optical performance and DM statistics for one AO step."""
        saturation_limit = 0.95 * float(max_command)
        stats = self._command_statistics(
            command,
            previous_command,
            saturation_limit,
        )

        self.records.append({
            "phase": phase,
            "type": "env",

            "episode": int(episode),
            "episode_step": int(episode_step),
            "global_env_step": int(global_env_step),
            "gradient_step": np.nan,

            # Optical performance
            "strehl": float(strehl),
            "le_strehl": (
                float(le_strehl)
                if le_strehl is not None and np.isfinite(le_strehl)
                else np.nan
            ),

            # Optimization statistics are not defined on environment rows
            "critic_loss": np.nan,
            "actor_loss": np.nan,
            "alpha_loss": np.nan,
            "alpha": np.nan,
            "entropy": np.nan,
            "policy_std": np.nan,

            # DM command statistics
            "command_min": stats[0],
            "command_q25": stats[1],
            "command_median": stats[2],
            "command_q75": stats[3],
            "command_max": stats[4],
            "command_abs_max": stats[5],
            "command_rms": stats[6],
            "saturation_fraction": stats[7],
            "delta_command_rms": stats[8],
        })

    def should_log_gradient(self, gradient_step):
        return int(gradient_step) % self.gradient_log_every == 0

    def log_gradient_step(
        self, *,
        phase,
        episode,
        gradient_step,
        critic_loss,
        actor_loss,
        alpha_loss,
        alpha,
        entropy,
        policy_std,
    ):
        """Log SAC optimization scalars with one accelerator->CPU transfer."""
        with torch.no_grad():
            values = torch.stack([
                critic_loss.detach().reshape(()),
                actor_loss.detach().reshape(()),
                alpha_loss.detach().reshape(()),
                alpha.detach().reshape(()),
                entropy.detach().reshape(()),
                policy_std.detach().reshape(()),
            ]).float().cpu().numpy()

        self.records.append({
            "phase": phase,
            "type": "gradient",

            "episode": int(episode),
            "episode_step": np.nan,
            "global_env_step": np.nan,
            "gradient_step": int(gradient_step),

            "strehl": np.nan,
            "le_strehl": np.nan,

            "critic_loss": float(values[0]),
            "actor_loss": float(values[1]),
            "alpha_loss": float(values[2]),
            "alpha": float(values[3]),
            "entropy": float(values[4]),
            "policy_std": float(values[5]),

            "command_min": np.nan,
            "command_q25": np.nan,
            "command_median": np.nan,
            "command_q75": np.nan,
            "command_max": np.nan,
            "command_abs_max": np.nan,
            "command_rms": np.nan,
            "saturation_fraction": np.nan,
            "delta_command_rms": np.nan,
        })

    def dataframe(self):
        if not self.records:
            return pd.DataFrame(columns=self.COLUMNS)
        return pd.DataFrame.from_records(
            self.records,
            columns=self.COLUMNS,
        )

    def save_csv(self, path):
        """Compatibility helper: save the currently buffered records."""
        self._atomic_csv_write(self.dataframe(), path)

    def strehl_summary(self, phase=None):
        """Compatibility helper for summaries of currently buffered records."""
        return self.strehl_summary_from_df(
            self.dataframe(),
            phase=phase,
        )


    @staticmethod
    def _atomic_csv_write(df, path):
        path = os.fspath(path)
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        tmp_path = path + ".tmp"
        df.to_csv(tmp_path, index=False)
        os.replace(tmp_path, path)

    def flush_shard(self, directory, shard_id):
        """Persist current buffered records and then free them from RAM.

        ``shard_id`` is deterministic. If a Colab VM dies after the shard is
        written but before the matching checkpoint is committed, rerunning the
        same segment simply overwrites the same shard rather than duplicating it.
        """
        if not self.records:
            return None

        os.makedirs(directory, exist_ok=True)
        shard_path = os.path.join(
            directory,
            f"part_{str(shard_id)}.csv",
        )

        self._atomic_csv_write(self.dataframe(), shard_path)
        self.records.clear()
        return shard_path

    @staticmethod
    def clear_shards(directory):
        if os.path.isdir(directory):
            shutil.rmtree(directory)
        os.makedirs(directory, exist_ok=True)

    @classmethod
    def load_shards(cls, directory):
        paths = sorted(glob.glob(os.path.join(directory, "part_*.csv")))
        if not paths:
            return pd.DataFrame(columns=cls.COLUMNS)

        frames = [pd.read_csv(path) for path in paths]
        df = pd.concat(frames, ignore_index=True)

        # Restore a deterministic temporal order after a resumed run.
        type_order = df["type"].map({"env": 0, "gradient": 1}).fillna(2)
        sort_env = df["global_env_step"].fillna(np.inf)
        sort_grad = df["gradient_step"].fillna(np.inf)

        df = (
            df.assign(
                _type_order=type_order,
                _sort_env=sort_env,
                _sort_grad=sort_grad,
            )
            .sort_values(
                ["episode", "_type_order", "_sort_env", "_sort_grad"],
                kind="stable",
            )
            .drop(columns=["_type_order", "_sort_env", "_sort_grad"])
            .reset_index(drop=True)
        )

        return df

    @classmethod
    def finalize_shards(cls, directory, final_csv_path):
        """Merge all persisted shards into the complete final raw log."""
        df = cls.load_shards(directory)
        cls._atomic_csv_write(df, final_csv_path)
        return df

    @staticmethod
    def strehl_summary_from_df(df, phase=None):
        env_df = df[df["type"] == "env"].copy()
        if phase is not None:
            env_df = env_df[env_df["phase"] == phase]

        if env_df.empty:
            return pd.DataFrame()

        return (
            env_df
            .groupby(["phase", "episode"], as_index=False)["strehl"]
            .agg(["mean", "std", "median", "min", "max"])
        )

    @staticmethod
    def le_strehl_episode_from_df(df, phase=None):
        """Return the last available LE Strehl for each episode.

        Since the final environment step is forced to compute LE Strehl, this
        gives the actual end-of-episode long-exposure value.
        """
        env_df = df[df["type"] == "env"].copy()
        if phase is not None:
            env_df = env_df[env_df["phase"] == phase]

        env_df = env_df[np.isfinite(env_df["le_strehl"])]
        if env_df.empty:
            return pd.DataFrame(
                columns=["phase", "episode", "le_strehl_final"]
            )

        out = (
            env_df
            .sort_values(["phase", "episode", "episode_step"])
            .groupby(["phase", "episode"], as_index=False)
            .tail(1)[["phase", "episode", "le_strehl"]]
            .rename(columns={"le_strehl": "le_strehl_final"})
            .reset_index(drop=True)
        )
        return out

    @classmethod
    def le_strehl_summary_from_df(cls, df, phase=None):
        per_episode = cls.le_strehl_episode_from_df(df, phase=phase)
        if per_episode.empty:
            return pd.DataFrame()

        return (
            per_episode
            .groupby("phase", as_index=False)["le_strehl_final"]
            .agg(["count", "mean", "std", "median", "min", "max"])
        )

    @classmethod
    def save_statistics_bundle(cls, df, output_dir, prefix):
        """Save the raw log plus the agreed aggregate optical statistics."""
        os.makedirs(output_dir, exist_ok=True)

        raw_path = os.path.join(output_dir, f"{prefix}_log.csv")
        sr_path = os.path.join(output_dir, f"{prefix}_strehl_by_episode.csv")
        le_episode_path = os.path.join(
            output_dir,
            f"{prefix}_le_strehl_by_episode.csv",
        )
        le_summary_path = os.path.join(
            output_dir,
            f"{prefix}_le_strehl_summary.csv",
        )

        cls._atomic_csv_write(df, raw_path)

        sr_summary = cls.strehl_summary_from_df(df)
        le_episode = cls.le_strehl_episode_from_df(df)
        le_summary = cls.le_strehl_summary_from_df(df)

        cls._atomic_csv_write(sr_summary, sr_path)
        cls._atomic_csv_write(le_episode, le_episode_path)
        cls._atomic_csv_write(le_summary, le_summary_path)

        return {
            "raw_log": raw_path,
            "strehl_by_episode": sr_path,
            "le_strehl_by_episode": le_episode_path,
            "le_strehl_summary": le_summary_path,
        }


### SAC Class

In [108]:
import os
import random


class SAC():
    """Soft Actor-Critic controller with CUDA-aware training, logging
    and rare resumable checkpoints.
    """

    def __init__(self, args):
        (
            self.env,
            self.actor,
            self.critic,
            self.target_critic,
            self.log_alpha,
            self.replay,
            self.actor_optim,
            self.critic_optim,
            self.alpha_optim,
            self.device
        ) = initialise(args)

        self.args = args

        self.episodes = args.episodes
        self.train_each_eval = args.train_each_eval
        self.episode_steps = args.episode_steps
        self.warm_up_steps = args.warm_up_steps
        self.env_steps = args.env_steps
        self.gradient_steps = args.gradient_steps
        self.batch_size = args.batch_size
        self.minibatch_size = args.minibatch_size
        self.target_entropy = args.target_entropy

        self.checkpoint_every = args.checkpoint_every

        self.checkpoint_dir = args.checkpoint_dir

        self.checkpoint_replay_limit = args.checkpoint_replay_limit

        self.pending_state = None
        self.pending_action = None

        self.current_episode = 0
        self.current_episode_step = 0

        self.global_env_step = 0
        self.global_gradient_step = 0

        self.phase = "train"
        self.previous_command = None

        if args.save_log:
          self.logger = RLLogger(
              gradient_log_every= args.log_gradient_every, #getattr(args, "log_gradient_every", 10),
              le_strehl_every= args.le_strehl_every #getattr(args, "le_strehl_every", 1),
          )

        if cuda.is_available():
            print('Running on gpu')
        else:
            print('Running on cpu')

        # [v2] empirical calibration of the normalisation scales (classical integrator, no RL)
        self.env.calibrate_scales()

        # [v2] mask of the illuminated actuators as a (1,1,H,W) tensor, in the same (flipped) frame as the state
        _side = int(np.sqrt(self.env.num_modes))
        _mask2d = np.flip(np.asarray(self.env.mask, dtype=np.float32).reshape(_side, _side), axis=0).copy()
        self.mask_t = torch.as_tensor(_mask2d, device=self.device).reshape(1, 1, _side, _side)
        self.best_eval_le = -np.inf


    # ==============================================================
    # ENVIRONMENT
    # ==============================================================

    def warm_up(self):
        for _ in range(self.warm_up_steps):
            self.env.compute_reconstruction()
            self.env.apply_DM_command()


    def _current_le_strehl(self, force=False):

        every = self.args.le_strehl_every

        if every <= 0 and not force:
            return np.nan

        if (
            not force
            and self.current_episode_step % every != 0
        ):
            return np.nan

        return self.env.get_long_exposure_strehl()

    # ==============================================================
    # ENVIRONMENT LOOP
    # ==============================================================


    def environment_loop(self, train=True):

        for _ in range(self.env_steps):

            self.env.compute_reconstruction()

            # SAC exploration is stochastic only during training.
            # Evaluation uses deterministic tanh(mean(s)).
            rew, state, action = rl_control_step(
                env=self.env,
                actor=self.actor,
                device=self.device,
                deterministic=not train,
            )

            command = np.asarray(self.env.commands[-1]).copy()

            if self.args.save_log:
              self.logger.log_environment_step(
                  phase=self.phase,

                  episode=self.current_episode,
                  episode_step=self.current_episode_step,

                  global_env_step=self.global_env_step,

                  strehl=self.env.strehl_list[-1],

                  le_strehl=self._current_le_strehl(
                      force=(
                          train
                          and self.current_episode_step
                          == (
                              self.episode_steps
                              * self.env_steps
                              - 1
                          )
                      )
                      or (not train)
                  ),

                  command=command,
                  max_command=self.env.max_command,
                  previous_command=self.previous_command,
              )

            self.previous_command = command

            self.current_episode_step += 1
            self.global_env_step += 1

            if train:

                if (
                    self.pending_state is not None
                    and self.pending_action is not None
                ):
                    self.replay.append(
                        (
                            self.pending_state,
                            self.pending_action,
                            rew,
                            state
                        )
                    )

                self.pending_state = state
                self.pending_action = action


    # ==============================================================
    # GRADIENT LOOP
    # ==============================================================

    def gradient_loop(self):

        self.actor.train()
        self.critic.train()

        for _ in range(self.gradient_steps):

            for minibatch in self.replay.sample_batch_iterator(
                batch_size=self.batch_size,
                minibatch_size=self.minibatch_size,
            ):

                states, actions, rewards, next_states = (
                    minibatch_to_torch(
                        *minibatch,
                        device=self.device
                    )
                )

                critic_loss = self.update_critic(
                    states,
                    actions,
                    rewards,
                    next_states,
                    self.log_alpha
                )

                (
                    log_pis,
                    actor_loss,
                    policy_std
                ) = self.update_actor(
                    states,
                    self.log_alpha
                )

                alpha_loss = self.update_alpha(
                    self.log_alpha,
                    log_pis,
                    self.target_entropy
                )

                self.update_target_critic()

                if (
                    self.args.save_log and
                    self.logger.should_log_gradient(self.global_gradient_step)
                    ):

                    self.logger.log_gradient_step(
                        phase=self.phase,

                        episode=self.current_episode,

                        gradient_step=self.global_gradient_step,

                        critic_loss=critic_loss,
                        actor_loss=actor_loss,
                        alpha_loss=alpha_loss,

                        alpha=self.log_alpha.exp(),

                        entropy=-self._masked_mean(log_pis.detach()),

                        policy_std=policy_std,
                    )

                self.global_gradient_step += 1


    # ==============================================================
    # CRITIC
    # ==============================================================

    def _masked_mean(self, x):
        """[v2] Mean over the illuminated actuators (and over the batch) of a (B,1,H,W) tensor."""
        return (x * self.mask_t).sum() / (self.mask_t.sum() * x.shape[0])

    def compute_critic_loss(
        self,
        qvalues1,
        qvalues2,
        qvalues_target,
        rewards,
        alpha,
        next_log_pi,
        gamma
    ):

        target = (rewards + gamma * (qvalues_target - alpha * next_log_pi))

        loss_q1 = 0.5 * (qvalues1 - target).pow(2)

        loss_q2 = 0.5 * (qvalues2 - target).pow(2)

        return (self._masked_mean(loss_q1) + self._masked_mean(loss_q2))


    def update_critic(
        self,
        states,
        actions,
        rewards,
        next_states,
        log_alpha,
        gamma=None
    ):

        if gamma is None:
            gamma = self.args.gamma

        alpha = log_alpha.exp().detach()

        with torch.no_grad():
            next_actions, next_log_pi = (self.actor.sample(next_states))
            next_actions = next_actions * self.mask_t  # [v2]

            q1_target, q2_target = (self.target_critic(next_states, next_actions))

            q_target = torch.minimum(q1_target, q2_target)

        q1, q2 = self.critic(states, actions)

        critic_loss = self.compute_critic_loss(
            q1,
            q2,
            q_target,
            rewards,
            alpha,
            next_log_pi,
            gamma
        )

        self.critic_optim.zero_grad(set_to_none=True)

        critic_loss.backward()

        self.critic_optim.step()

        return critic_loss.detach()


    # ==============================================================
    # ACTOR
    # ==============================================================

    def compute_actor_loss(
        self,
        log_pis,
        qvalues,
        alpha
    ):
        return self._masked_mean(alpha * log_pis - qvalues)


    def update_actor(
        self,
        states,
        log_alpha
    ):

        alpha = log_alpha.exp().detach()

        actions, log_pis = (self.actor.sample(states))
        actions = actions * self.mask_t  # [v2]

        # Need dQ/da but NOT gradients for critic parameters.
        self.critic.requires_grad_(False)

        q1, q2 = self.critic(states, actions)

        qvalues = torch.minimum(q1, q2)

        # [v2] BUG FIX: pass alpha, not log_alpha
        actor_loss = self.compute_actor_loss(log_pis, qvalues, alpha)

        self.actor_optim.zero_grad(set_to_none=True)

        actor_loss.backward()

        self.actor_optim.step()

        self.critic.requires_grad_(True)

        policy_std = (self.actor.log_std.detach().exp().mean())

        return (log_pis.detach(), actor_loss.detach(), policy_std)


    # ==============================================================
    # ALPHA
    # ==============================================================

    def compute_alpha_loss(
        self,
        log_pis,
        log_alpha,
        target_entropy
    ):

        return -self._masked_mean(log_alpha * (log_pis + target_entropy).detach())


    def update_alpha(
        self,
        log_alpha,
        log_pis,
        target_entropy
    ):

        alpha_loss = self.compute_alpha_loss(log_pis.detach(), log_alpha, target_entropy)

        self.alpha_optim.zero_grad(set_to_none=True)

        alpha_loss.backward()

        self.alpha_optim.step()

        return alpha_loss.detach()


    # ==============================================================
    # TARGET CRITIC
    # ==============================================================

    @torch.no_grad()
    def update_target_critic(self, tau=None):

        if tau is None:
            tau = self.args.tau

        for target_param, param in zip(
            self.target_critic.parameters(),
            self.critic.parameters()
        ):
            target_param.lerp_(
                param,
                tau
            )


    # ==============================================================
    # TRAIN EPISODE
    # ==============================================================

    def run_train_episode(self):

        self.warm_up()

        for _ in range(
            self.episode_steps
        ):

            self.environment_loop(
                train=True
            )

            if len(self.replay) >= self.batch_size:
                self.gradient_loop()


    # ==============================================================
    # SAVE POLICY
    # ==============================================================

    def save_policy(self, path: str):

        # Save CPU tensors for a small portable final policy.
        state_dict_cpu = {
            k: v.detach().cpu()
            for k, v
            in self.actor.state_dict().items()
        }

        torch.save(
            state_dict_cpu,
            path
        )

        # [v2] the policy is only meaningful together with the scales used to normalise its inputs
        import json
        with open(path + ".scales.json", "w") as f:
            json.dump({"rec_scale": float(self.env.rec_scale),
                       "cmd_scale": float(self.env.cmd_scale)}, f)


    # ==============================================================
    # CHECKPOINT UTILITIES
    # ==============================================================

    @staticmethod
    def _to_cpu_tree(obj):
        """
        Recursively move all tensors to CPU before serializing
        a checkpoint.
        """

        if torch.is_tensor(obj):
            return obj.detach().cpu()

        if isinstance(obj, dict):
            return {
                k: SAC._to_cpu_tree(v)
                for k, v in obj.items()
            }

        if isinstance(obj, list):
            return [
                SAC._to_cpu_tree(v)
                for v in obj
            ]

        if isinstance(obj, tuple):
            return tuple(
                SAC._to_cpu_tree(v)
                for v in obj
            )

        return obj


    @staticmethod
    def _optimizer_to_device(
        optimizer,
        device
    ):
        """
        Move optimizer state tensors to the current training device.
        """

        for state in optimizer.state.values():

            for key, value in state.items():

                if torch.is_tensor(value):

                    state[key] = value.to(
                        device
                    )


    # ==============================================================
    # SAVE CHECKPOINT
    # ==============================================================

    def save_checkpoint(
        self,
        path,
        *,
        next_episode,
        train_episodes_completed
    ):
        """
        Save one resumable checkpoint.

        Only the most recent checkpoint_replay_limit replay transitions
        are included by default in order to bound file size and I/O.

        Set checkpoint_replay_limit <= 0 for the full replay buffer.
        """

        os.makedirs(
            os.path.dirname(path) or ".",
            exist_ok=True
        )

        checkpoint = {

            "checkpoint_version": 2,

            "rec_scale": float(self.env.rec_scale),
            "cmd_scale": float(self.env.cmd_scale),

            # ------------------------------------------------------
            # Training progress
            # ------------------------------------------------------

            "next_episode":
                int(next_episode),

            "train_episodes_completed":
                int(train_episodes_completed),

            "global_env_step":
                int(self.global_env_step),

            "global_gradient_step":
                int(self.global_gradient_step),


            # ------------------------------------------------------
            # Networks
            # ------------------------------------------------------

            "actor":
                self._to_cpu_tree(
                    self.actor.state_dict()
                ),

            "critic":
                self._to_cpu_tree(
                    self.critic.state_dict()
                ),

            "target_critic":
                self._to_cpu_tree(
                    self.target_critic.state_dict()
                ),


            # ------------------------------------------------------
            # Alpha
            # ------------------------------------------------------

            "log_alpha":
                self.log_alpha
                .detach()
                .cpu(),


            # ------------------------------------------------------
            # Optimizers
            # ------------------------------------------------------

            "actor_optim":
                self._to_cpu_tree(
                    self.actor_optim.state_dict()
                ),

            "critic_optim":
                self._to_cpu_tree(
                    self.critic_optim.state_dict()
                ),

            "alpha_optim":
                self._to_cpu_tree(
                    self.alpha_optim.state_dict()
                ),


            # ------------------------------------------------------
            # Replay buffer
            # ------------------------------------------------------

            "replay":
                self.replay.state_dict(
                    max_items=
                    self.checkpoint_replay_limit
                ),


            # ------------------------------------------------------
            # Logger configuration
            # ------------------------------------------------------

            "logger_gradient_log_every":
                None,

            "logger_le_strehl_every":
                None,

            # ------------------------------------------------------
            # Random generator states
            # ------------------------------------------------------

            "python_rng_state":
                random.getstate(),

            "numpy_rng_state":
                np.random.get_state(),


            # torch.set_rng_state expects CPU ByteTensor.
            "torch_rng_state":
                torch.get_rng_state()
                .cpu()
                .to(torch.uint8),


            # Store CUDA RNG state representations as CPU uint8 tensors.
            "cuda_rng_state_all": (
                [
                    state
                    .cpu()
                    .to(torch.uint8)

                    for state
                    in torch.cuda.get_rng_state_all()
                ]

                if torch.cuda.is_available()

                else None
            ),
        }

        if self.args.save_log:
            checkpoint["logger_gradient_log_every"] = self.logger.gradient_log_every
            checkpoint["logger_le_strehl_every"] = self.logger.le_strehl_every

        # ----------------------------------------------------------
        # Atomic checkpoint writing
        # ----------------------------------------------------------

        # Save to temporary file first.
        #
        # If Colab dies during torch.save(), the previous checkpoint
        # remains intact.
        tmp_path = (
            path
            + ".tmp"
        )

        torch.save(
            checkpoint,
            tmp_path
        )

        os.replace(
            tmp_path,
            path
        )


        print(
            f"Checkpoint saved: {path} | "
            f"train episodes="
            f"{train_episodes_completed} | "
            f"replay transitions="
            f"{len(checkpoint['replay']['storage'])}"
        )


    # ==============================================================
    # LOAD CHECKPOINT
    # ==============================================================

    def load_checkpoint(self, path):
        """
        Restore complete SAC state.

        Returns
        -------
        next_episode : int
            Outer episode index from which training resumes.

        train_episodes_completed : int
            Number of training episodes already completed.
        """

        # ----------------------------------------------------------
        # IMPORTANT:
        # load EVERYTHING on CPU first.
        #
        # Networks, alpha and optimizer states are subsequently
        # moved explicitly to the correct device.
        #
        # This also prevents RNG tensors from being automatically
        # remapped to CUDA.
        # ----------------------------------------------------------

        try:

            checkpoint = torch.load(
                path,
                map_location="cpu",
                weights_only=False,
            )

        except TypeError:

            # Compatibility with older PyTorch versions.
            checkpoint = torch.load(
                path,
                map_location="cpu"
            )


        # ----------------------------------------------------------
        # Networks
        # ----------------------------------------------------------

        # [v2] checkpoints from the old code used a different reward/state normalisation
        if int(checkpoint.get("checkpoint_version", 1)) < 2:
            raise RuntimeError(
                f"{path} was created by the previous version of the code (different reward/state "
                "normalisation). Delete/rename it or set RESUME_FROM = None."
            )
        self.env.rec_scale = float(checkpoint["rec_scale"])
        self.env.cmd_scale = float(checkpoint["cmd_scale"])

        self.actor.load_state_dict(
            checkpoint["actor"]
        )

        self.critic.load_state_dict(
            checkpoint["critic"]
        )

        self.target_critic.load_state_dict(
            checkpoint["target_critic"]
        )

        self.target_critic.requires_grad_(
            False
        )


        # ----------------------------------------------------------
        # Alpha
        # ----------------------------------------------------------

        with torch.no_grad():

            self.log_alpha.copy_(
                checkpoint["log_alpha"]
                .to(self.device)
            )


        # ----------------------------------------------------------
        # Optimizers
        # ----------------------------------------------------------

        self.actor_optim.load_state_dict(
            checkpoint["actor_optim"]
        )

        self.critic_optim.load_state_dict(
            checkpoint["critic_optim"]
        )

        self.alpha_optim.load_state_dict(
            checkpoint["alpha_optim"]
        )


        # Optimizer states were deliberately loaded on CPU.
        # Move them explicitly to the current training device.

        self._optimizer_to_device(
            self.actor_optim,
            self.device
        )

        self._optimizer_to_device(
            self.critic_optim,
            self.device
        )

        self._optimizer_to_device(
            self.alpha_optim,
            self.device
        )


        # ----------------------------------------------------------
        # Replay buffer
        # ----------------------------------------------------------

        self.replay.load_state_dict(
            checkpoint["replay"]
        )


        # ----------------------------------------------------------
        # Global counters
        # ----------------------------------------------------------

        self.global_env_step = int(
            checkpoint.get(
                "global_env_step",
                0
            )
        )

        self.global_gradient_step = int(
            checkpoint.get(
                "global_gradient_step",
                0
            )
        )


        # ----------------------------------------------------------
        # Logger
        # ----------------------------------------------------------

        # Raw logs are persisted separately in CSV shards.
        # They are intentionally not duplicated in the checkpoint.
        if self.args.save_log:
          self.logger.records.clear()

          self.logger.gradient_log_every = int(
              checkpoint.get(
                  "logger_gradient_log_every",
                  self.logger.gradient_log_every
              )
          )

          self.logger.le_strehl_every = int(
              checkpoint.get(
                  "logger_le_strehl_every",
                  self.logger.le_strehl_every
              )
          )


        # ----------------------------------------------------------
        # Restore random streams
        # ----------------------------------------------------------

        # Python
        python_rng_state = checkpoint.get(
            "python_rng_state"
        )

        if python_rng_state is not None:

            random.setstate(
                python_rng_state
            )


        # NumPy
        numpy_rng_state = checkpoint.get(
            "numpy_rng_state"
        )

        if numpy_rng_state is not None:

            np.random.set_state(
                numpy_rng_state
            )


        # PyTorch CPU RNG
        torch_rng_state = checkpoint.get(
            "torch_rng_state"
        )

        if torch_rng_state is not None:

            torch_rng_state = (
                torch_rng_state
                .cpu()
                .to(torch.uint8)
            )

            torch.set_rng_state(
                torch_rng_state
            )


        # PyTorch CUDA RNG
        cuda_rng_state_all = checkpoint.get(
            "cuda_rng_state_all"
        )

        if (
            self.device.type == "cuda"
            and cuda_rng_state_all is not None
        ):

            cuda_rng_states = [

                state
                .cpu()
                .to(torch.uint8)

                for state
                in cuda_rng_state_all
            ]

            torch.cuda.set_rng_state_all(
                cuda_rng_states
            )


        # ----------------------------------------------------------
        # Resume position
        # ----------------------------------------------------------

        next_episode = int(
            checkpoint.get(
                "next_episode",
                0
            )
        )

        train_episodes_completed = int(
            checkpoint.get(
                "train_episodes_completed",
                0
            )
        )


        print(
            f"Checkpoint loaded: {path} | "
            f"resume outer episode="
            f"{next_episode} | "
            f"completed train episodes="
            f"{train_episodes_completed} | "
            f"replay transitions="
            f"{len(self.replay)}"
        )


        return (
            next_episode,
            train_episodes_completed
        )


    # ==============================================================
    # RESET
    # ==============================================================

    def reset(self):

        self.env.reset()

        self.pending_state = None
        self.pending_action = None

        self.current_episode_step = 0
        self.previous_command = None


    # ==============================================================
    # TRAIN / EVALUATION
    # ==============================================================

    def train_eval(
        self,
        policy_path: str = "sac_policy.pth",
        log_path: str = None,
        resume_from: str = None,
    ):
        """
        Train/evaluate SAC and return the complete raw statistics
        dataframe.

        Long runs do not keep the whole log in RAM.

        Records are flushed only at checkpoint boundaries and once
        at the end.

        The final dataframe is reconstructed from the persisted CSV
        shards.
        """

        tot_episodes = (
            self.episodes
            + (
                self.episodes
                // self.train_each_eval
            )
        )


        if resume_from is None:

            resume_from = (
                getattr(
                    self.args,
                    "resume_checkpoint",
                    ""
                )
                or None
            )


        if log_path is None:

            log_path = os.path.join(
                self.checkpoint_dir,
                "training_log.csv",
            )


        log_root, _ = os.path.splitext(
            log_path
        )

        log_parts_dir = (
            log_root
            + "_parts"
        )


        start_episode = 0
        train_episodes = 0
        eval_episodes = 0


        # ----------------------------------------------------------
        # Resume
        # ----------------------------------------------------------

        if resume_from is not None:

            (
                start_episode,
                train_episodes
            ) = self.load_checkpoint(
                resume_from
            )

            eval_episodes = max(
                0,
                start_episode
                - train_episodes
            )


            if not os.path.isdir(
                log_parts_dir
            ):

                print(
                    "WARNING: checkpoint loaded but no persisted "
                    "log shards were found at "
                    f"{log_parts_dir}. "
                    "Training can resume, but statistics "
                    "from before the checkpoint are unavailable."
                )


        # ----------------------------------------------------------
        # Fresh run
        # ----------------------------------------------------------

        else:
            if self.args.save_log:
              # Prevent old shards from contaminating a new experiment.
              self.logger.clear_shards(
                  log_parts_dir
              )


        checkpoint_path = os.path.join(
            self.checkpoint_dir,
            "latest_checkpoint.pth",
        )


        # ----------------------------------------------------------
        # Main episodes
        # ----------------------------------------------------------

        for episode in tqdm(
            range(
                start_episode,
                tot_episodes
            ),
            initial=start_episode,
            total=tot_episodes,
        ):

            self.current_episode = episode

            self.reset()


            is_eval_episode = (
                (episode + 1)
                % (
                    self.train_each_eval
                    + 1
                )
                == 0
            )


            # ======================================================
            # EVALUATION
            # ======================================================

            if is_eval_episode:

                self.phase = "eval"

                self.actor.eval()

                eval_episodes += 1


                print(
                    f"\n[EVAL | RL] "
                    f"Episode {episode + 1}/"
                    f"{tot_episodes} "
                    f"(evaluation "
                    f"#{eval_episodes})"
                )


                self.warm_up()


                # [v2] Deterministic evaluation over a FULL episode (it used to be a single step),
                # keeping the best policy seen so far.
                for _ in range(self.episode_steps):
                    self.environment_loop(
                        train=False
                    )

                eval_le = self.env.get_long_exposure_strehl()
                print(f"[EVAL | RL] final long-exposure Strehl = {eval_le:.4f}")
                if np.isfinite(eval_le) and eval_le > self.best_eval_le:
                    self.best_eval_le = eval_le
                    self.save_policy(os.path.splitext(policy_path)[0] + "_best.pth")


            # ======================================================
            # TRAINING
            # ======================================================

            else:

                self.phase = "train"

                self.actor.train()


                print(
                    f"\n[TRAIN | RL] "
                    f"Episode {episode + 1}/"
                    f"{tot_episodes} "
                    f"(training "
                    f"#{train_episodes + 1}/"
                    f"{self.episodes})"
                )


                self.run_train_episode()


                train_episodes += 1


                print(
                    f"[TRAIN | RL] "
                    f"Completed training episode "
                    f"{train_episodes}/"
                    f"{self.episodes}"
                )


                # --------------------------------------------------
                # Rare checkpoint
                # --------------------------------------------------

                if (
                    self.checkpoint_every > 0

                    and

                    train_episodes
                    % self.checkpoint_every
                    == 0
                ):

                    # Persist statistics BEFORE checkpoint.
                    #
                    # Deterministic shard name means that if Colab
                    # dies between shard writing and checkpoint
                    # commit, the same segment is safely overwritten
                    # after resume.
                    if self.args.save_log:
                      self.logger.flush_shard(
                          log_parts_dir,
                          f"{train_episodes:06d}",
                      )


                    self.save_checkpoint(
                        checkpoint_path,

                        next_episode=
                            episode + 1,

                        train_episodes_completed=
                            train_episodes,
                    )


        # ----------------------------------------------------------
        # Persist tail after final checkpoint
        # ----------------------------------------------------------
        if self.args.save_log:
          self.logger.flush_shard(
              log_parts_dir,
              f"final_{train_episodes:06d}",
          )


        # ----------------------------------------------------------
        # Final policy
        # ----------------------------------------------------------

        self.save_policy(
            policy_path
        )


        # ----------------------------------------------------------
        # Reconstruct complete log
        # ----------------------------------------------------------
        if self.args.save_log:
          training_df = (
              self.logger.finalize_shards(
                  log_parts_dir,
                  log_path,
              )
          )
          print(
            f"Complete training log "
            f"saved to "
            f"{log_path}"
          )

        else:
          training_df = None


        print(
            f"Training completed. "
            f"Policy saved to "
            f"{policy_path}"
        )


        return training_df

In [109]:
# import os
# import random


# class SAC():
#     """Soft Actor-Critic controller with CUDA-aware training, logging and rare resumable checkpoints."""

#     def __init__(self, args):
#         (self.env, self.actor, self.critic, self.target_critic, self.log_alpha,
#          self.replay, self.actor_optim, self.critic_optim, self.alpha_optim,
#          self.device) = initialise(args)

#         self.args = args
#         self.episodes = args.episodes
#         self.train_each_eval = args.train_each_eval
#         self.episode_steps = args.episode_steps
#         self.warm_up_steps = args.warm_up_steps
#         self.env_steps = args.env_steps
#         self.gradient_steps = args.gradient_steps
#         self.batch_size = args.batch_size
#         self.minibatch_size = args.minibatch_size

#         self.checkpoint_every = int(getattr(args, "checkpoint_every", 20))
#         self.checkpoint_dir = str(getattr(args, "checkpoint_dir", "checkpoints"))
#         self.checkpoint_replay_limit = int(getattr(args, "checkpoint_replay_limit", 2000))

#         self.pending_state = None
#         self.pending_action = None

#         self.current_episode = 0
#         self.current_episode_step = 0
#         self.global_env_step = 0
#         self.global_gradient_step = 0
#         self.phase = "train"
#         self.previous_command = None

#         self.logger = RLLogger(
#             gradient_log_every=getattr(args, "log_gradient_every", 10),
#             le_strehl_every=getattr(args, "le_strehl_every", 1),
#         )

#         print(f"PyTorch device: {self.device}")

#     def warm_up(self):
#         for _ in range(self.warm_up_steps):
#             self.env.compute_reconstruction()
#             self.env.apply_DM_command()

#     def _current_le_strehl(self, force=False):
#         every = self.logger.le_strehl_every
#         if every <= 0 and not force:
#             return np.nan
#         if not force and self.current_episode_step % every != 0:
#             return np.nan
#         return self.env.get_long_exposure_strehl()

#     def environment_loop(self, train=True):
#         for _ in range(self.env_steps):
#             self.env.compute_reconstruction()

#             # SAC exploration is stochastic only during training.
#             # Evaluation uses the deterministic tanh(mean(s)) policy.
#             rew, state, action = rl_control_step(
#                 env=self.env,
#                 actor=self.actor,
#                 device=self.device,
#                 deterministic=not train,
#             )

#             command = np.asarray(self.env.commands[-1]).copy()
#             self.logger.log_environment_step(
#                 phase=self.phase,
#                 episode=self.current_episode,
#                 episode_step=self.current_episode_step,
#                 global_env_step=self.global_env_step,
#                 strehl=self.env.strehl_list[-1],
#                 le_strehl=self._current_le_strehl(
#                     force=(
#                         train
#                         and self.current_episode_step
#                         == (self.episode_steps * self.env_steps - 1)
#                     )
#                     or (not train)
#                 ),
#                 command=command,
#                 previous_command=self.previous_command,
#                 max_command=self.env.max_command,
#             )
#             self.previous_command = command
#             self.current_episode_step += 1
#             self.global_env_step += 1

#             if train:
#                 if self.pending_state is not None and self.pending_action is not None:
#                     self.replay.append((self.pending_state, self.pending_action, rew, state))
#                 self.pending_state = state
#                 self.pending_action = action

#     def gradient_loop(self):
#         self.actor.train()
#         self.critic.train()

#         for _ in range(self.gradient_steps):
#             for minibatch in self.replay.sample_batch_iterator(
#                 batch_size=self.batch_size,
#                 minibatch_size=self.minibatch_size,
#             ):
#                 states, actions, rewards, next_states = minibatch_to_torch(
#                     *minibatch, device=self.device
#                 )

#                 critic_loss = self.update_critic(
#                     states, actions, rewards, next_states, self.log_alpha
#                 )
#                 log_pis, actor_loss, policy_std = self.update_actor(
#                     states, self.log_alpha
#                 )
#                 alpha_loss = self.update_alpha(self.log_alpha, log_pis)
#                 self.update_target_critic()

#                 if self.logger.should_log_gradient(self.global_gradient_step):
#                     self.logger.log_gradient_step(
#                         phase=self.phase,
#                         episode=self.current_episode,
#                         gradient_step=self.global_gradient_step,
#                         critic_loss=critic_loss,
#                         actor_loss=actor_loss,
#                         alpha_loss=alpha_loss,
#                         alpha=self.log_alpha.exp(),
#                         entropy=-log_pis.detach().mean(),
#                         policy_std=policy_std,
#                     )

#                 self.global_gradient_step += 1

#     def compute_critic_loss(self, qvalues1, qvalues2, qvalues_target,
#                             rewards, alpha, next_log_pi, gamma):
#         target = rewards + gamma * qvalues_target - alpha * next_log_pi
#         loss_q1 = 0.5 * (qvalues1 - target).pow(2)
#         loss_q2 = 0.5 * (qvalues2 - target).pow(2)
#         return loss_q1.mean() + loss_q2.mean()

#     def update_critic(self, states, actions, rewards, next_states,
#                       log_alpha, gamma=None):
#         if gamma is None:
#             gamma = self.args.gamma

#         alpha = log_alpha.exp().detach()
#         with torch.no_grad():
#             next_actions, next_log_pi = self.actor.sample(next_states)
#             q1_target, q2_target = self.target_critic(next_states, next_actions)
#             q_target = torch.minimum(q1_target, q2_target)

#         q1, q2 = self.critic(states, actions)
#         critic_loss = self.compute_critic_loss(
#             q1, q2, q_target, rewards, alpha, next_log_pi, gamma
#         )

#         self.critic_optim.zero_grad(set_to_none=True)
#         critic_loss.backward()
#         self.critic_optim.step()
#         return critic_loss.detach()

#     def compute_actor_loss(self, log_pis, qvalues, alpha):
#         return (alpha * log_pis - qvalues).mean()

#     def update_actor(self, states, log_alpha):
#         alpha = log_alpha.exp().detach()
#         actions, log_pis = self.actor.sample(states)

#         # We need dQ/da, but not gradients for critic parameters during the actor step.
#         self.critic.requires_grad_(False)
#         q1, q2 = self.critic(states, actions)
#         qvalues = torch.minimum(q1, q2)
#         actor_loss = self.compute_actor_loss(log_pis, qvalues, alpha)

#         self.actor_optim.zero_grad(set_to_none=True)
#         actor_loss.backward()
#         self.actor_optim.step()
#         self.critic.requires_grad_(True)

#         policy_std = self.actor.log_std.detach().exp().mean()
#         return log_pis.detach(), actor_loss.detach(), policy_std

#     def compute_alpha_loss(self, log_pis, alpha, target_entropy):
#         return -(alpha * (log_pis + target_entropy).detach()).mean()

#     def update_alpha(self, log_alpha, log_pis, target_entropy=-1.0):
#         alpha = log_alpha.exp()
#         alpha_loss = self.compute_alpha_loss(log_pis.detach(), alpha, target_entropy)

#         self.alpha_optim.zero_grad(set_to_none=True)
#         alpha_loss.backward()
#         self.alpha_optim.step()
#         return alpha_loss.detach()

#     @torch.no_grad()
#     def update_target_critic(self, tau=None):
#         if tau is None:
#             tau = self.args.tau
#         for target_param, param in zip(
#             self.target_critic.parameters(), self.critic.parameters()
#         ):
#             target_param.lerp_(param, tau)

#     def run_train_episode(self):
#         self.warm_up()
#         for _ in range(self.episode_steps):
#             self.environment_loop(train=True)
#             if len(self.replay) >= self.batch_size:
#                 self.gradient_loop()

#     def save_policy(self, path: str):
#         # Save CPU tensors for a small, portable final policy file.
#         state_dict_cpu = {k: v.detach().cpu() for k, v in self.actor.state_dict().items()}
#         torch.save(state_dict_cpu, path)

#     @staticmethod
#     def _to_cpu_tree(obj):
#         """Recursively move tensors to CPU before serializing a checkpoint."""
#         if torch.is_tensor(obj):
#             return obj.detach().cpu()
#         if isinstance(obj, dict):
#             return {k: SAC._to_cpu_tree(v) for k, v in obj.items()}
#         if isinstance(obj, list):
#             return [SAC._to_cpu_tree(v) for v in obj]
#         if isinstance(obj, tuple):
#             return tuple(SAC._to_cpu_tree(v) for v in obj)
#         return obj

#     @staticmethod
#     def _optimizer_to_device(optimizer, device):
#         for state in optimizer.state.values():
#             for key, value in state.items():
#                 if torch.is_tensor(value):
#                     state[key] = value.to(device)

#     def save_checkpoint(self, path, *, next_episode, train_episodes_completed):
#         """Save one resumable checkpoint.

#         Checkpoints are intentionally rare. By default only the most recent
#         ``checkpoint_replay_limit`` replay transitions are included, bounding
#         checkpoint size and I/O. Set that argument <= 0 for the full replay.
#         """
#         os.makedirs(os.path.dirname(path) or ".", exist_ok=True)

#         checkpoint = {
#             "checkpoint_version": 1,
#             "next_episode": int(next_episode),
#             "train_episodes_completed": int(train_episodes_completed),
#             "global_env_step": int(self.global_env_step),
#             "global_gradient_step": int(self.global_gradient_step),

#             "actor": self._to_cpu_tree(self.actor.state_dict()),
#             "critic": self._to_cpu_tree(self.critic.state_dict()),
#             "target_critic": self._to_cpu_tree(self.target_critic.state_dict()),
#             "log_alpha": self.log_alpha.detach().cpu(),

#             "actor_optim": self._to_cpu_tree(self.actor_optim.state_dict()),
#             "critic_optim": self._to_cpu_tree(self.critic_optim.state_dict()),
#             "alpha_optim": self._to_cpu_tree(self.alpha_optim.state_dict()),

#             "replay": self.replay.state_dict(
#                 max_items=self.checkpoint_replay_limit
#             ),

#             "logger_gradient_log_every": self.logger.gradient_log_every,
#             "logger_le_strehl_every": self.logger.le_strehl_every,

#             "python_rng_state": random.getstate(),
#             "numpy_rng_state": np.random.get_state(),
#             "torch_rng_state": torch.get_rng_state().cpu().to(torch.uint8),
#             "cuda_rng_state_all": (
#                 [s.cpu().to(torch.uint8) for s in torch.cuda.get_rng_state_all()]
#                 if torch.cuda.is_available()
#                 else None
#             ),
#         }

#         # Write to a temporary file first: if Colab dies during serialization,
#         # the previous latest_checkpoint.pth remains usable.
#         tmp_path = path + ".tmp"
#         torch.save(checkpoint, tmp_path)
#         os.replace(tmp_path, path)

#         print(
#             f"Checkpoint saved: {path} | "
#             f"train episodes={train_episodes_completed} | "
#             f"replay transitions={len(checkpoint['replay']['storage'])}"
#         )

#     def load_checkpoint(self, path):
#         """Restore SAC state and return (next_episode, train_episodes_completed)."""
#         try:
#             checkpoint = torch.load(
#                 path,
#                 map_location=self.device,
#                 weights_only=False,
#             )
#         except TypeError:
#             # Compatibility with older PyTorch versions.
#             checkpoint = torch.load(path, map_location=self.device)

#         self.actor.load_state_dict(checkpoint["actor"])
#         self.critic.load_state_dict(checkpoint["critic"])
#         self.target_critic.load_state_dict(checkpoint["target_critic"])
#         self.target_critic.requires_grad_(False)

#         with torch.no_grad():
#             self.log_alpha.copy_(checkpoint["log_alpha"].to(self.device))

#         self.actor_optim.load_state_dict(checkpoint["actor_optim"])
#         self.critic_optim.load_state_dict(checkpoint["critic_optim"])
#         self.alpha_optim.load_state_dict(checkpoint["alpha_optim"])
#         self._optimizer_to_device(self.actor_optim, self.device)
#         self._optimizer_to_device(self.critic_optim, self.device)
#         self._optimizer_to_device(self.alpha_optim, self.device)

#         self.replay.load_state_dict(checkpoint["replay"])

#         self.global_env_step = int(checkpoint.get("global_env_step", 0))
#         self.global_gradient_step = int(checkpoint.get("global_gradient_step", 0))

#         # Raw logs are persisted separately in rare CSV shards.
#         # They are intentionally not duplicated inside the .pth checkpoint.
#         self.logger.records.clear()
#         self.logger.gradient_log_every = int(
#             checkpoint.get("logger_gradient_log_every", self.logger.gradient_log_every)
#         )
#         self.logger.le_strehl_every = int(
#             checkpoint.get("logger_le_strehl_every", self.logger.le_strehl_every)
#         )

#         # Restore random streams after loading all serialized state.
#         if checkpoint.get("python_rng_state") is not None:
#             random.setstate(checkpoint["python_rng_state"])
#         if checkpoint.get("numpy_rng_state") is not None:
#             np.random.set_state(checkpoint["numpy_rng_state"])
#         if checkpoint.get("torch_rng_state") is not None:
#             # torch.set_rng_state expects a CPU tensor, so ensure it's on CPU and correct type.
#             torch.set_rng_state(checkpoint["torch_rng_state"].cpu().to(torch.uint8))
#         if (
#             self.device.type == "cuda"
#             and checkpoint.get("cuda_rng_state_all") is not None
#         ):
#             # Ensure each state is a ByteTensor (uint8) before setting and move to device.
#             cuda_rng_states_converted = [s.to(self.device).to(torch.uint8) for s in checkpoint["cuda_rng_state_all"]]
#             torch.cuda.set_rng_state_all(cuda_rng_states_converted)

#         next_episode = int(checkpoint.get("next_episode", 0))
#         train_episodes_completed = int(
#             checkpoint.get("train_episodes_completed", 0)
#         )

#         print(
#             f"Checkpoint loaded: {path} | "
#             f"resume outer episode={next_episode} | "
#             f"completed train episodes={train_episodes_completed} | "
#             f"replay transitions={len(self.replay)}"
#         )

#         return next_episode, train_episodes_completed

#     def reset(self):
#         self.env.reset()
#         self.pending_state = None
#         self.pending_action = None
#         self.current_episode_step = 0
#         self.previous_command = None

#     def train_eval(
#         self,
#         policy_path: str = "sac_policy.pth",
#         log_path: str = None,
#         resume_from: str = None,
#     ):
#         """Train/evaluate SAC and return the complete raw statistics dataframe.

#         Long runs do not keep the whole log in RAM. Records are flushed only at
#         checkpoint boundaries and once at the end. The final dataframe is rebuilt
#         from those shards, so all instantaneous Strehl and command statistics are
#         retained.
#         """
#         tot_episodes = self.episodes + (self.episodes // self.train_each_eval)

#         if resume_from is None:
#             resume_from = getattr(self.args, "resume_checkpoint", "") or None

#         if log_path is None:
#             log_path = os.path.join(
#                 self.checkpoint_dir,
#                 "training_log.csv",
#             )

#         log_root, _ = os.path.splitext(log_path)
#         log_parts_dir = log_root + "_parts"

#         start_episode = 0
#         train_episodes = 0
#         eval_episodes = 0

#         if resume_from is not None:
#             start_episode, train_episodes = self.load_checkpoint(resume_from)
#             eval_episodes = max(0, start_episode - train_episodes)

#             if not os.path.isdir(log_parts_dir):
#                 print(
#                     "WARNING: checkpoint loaded but no persisted log shards were "
#                     f"found at {log_parts_dir}. Training can resume, but statistics "
#                     "from before the checkpoint are unavailable."
#                 )
#         else:
#             # Fresh run: prevent stale shards from a previous experiment from
#             # contaminating the final dataframe.
#             self.logger.clear_shards(log_parts_dir)

#         checkpoint_path = os.path.join(
#             self.checkpoint_dir,
#             "latest_checkpoint.pth",
#         )

#         for episode in tqdm(
#             range(start_episode, tot_episodes),
#             initial=start_episode,
#             total=tot_episodes,
#         ):
#             self.current_episode = episode
#             self.reset()

#             is_eval_episode = (
#                 (episode + 1) % (self.train_each_eval + 1) == 0
#             )

#             if is_eval_episode:
#                 self.phase = "eval"
#                 self.actor.eval()
#                 eval_episodes += 1

#                 print(
#                     f"\n[EVAL | RL] Episode {episode + 1}/{tot_episodes} "
#                     f"(evaluation #{eval_episodes})"
#                 )

#                 self.warm_up()

#                 # Evaluation policy is deterministic.
#                 self.environment_loop(train=False)

#             else:
#                 self.phase = "train"
#                 self.actor.train()

#                 print(
#                     f"\n[TRAIN | RL] Episode {episode + 1}/{tot_episodes} "
#                     f"(training #{train_episodes + 1}/{self.episodes})"
#                 )

#                 self.run_train_episode()

#                 train_episodes += 1
#                 print(
#                     f"[TRAIN | RL] Completed training episode "
#                     f"{train_episodes}/{self.episodes}"
#                 )

#                 if (
#                     self.checkpoint_every > 0
#                     and train_episodes % self.checkpoint_every == 0
#                 ):
#                     # First make the statistics durable. The deterministic shard
#                     # name makes a crash between this write and checkpoint commit
#                     # safe: the same segment is overwritten after resume.
#                     self.logger.flush_shard(
#                         log_parts_dir,
#                         f"{train_episodes:06d}",
#                     )

#                     self.save_checkpoint(
#                         checkpoint_path,
#                         next_episode=episode + 1,
#                         train_episodes_completed=train_episodes,
#                     )

#         # Persist the tail after the last checkpoint (often the final eval episode).
#         self.logger.flush_shard(
#             log_parts_dir,
#             f"final_{train_episodes:06d}",
#         )

#         self.save_policy(policy_path)

#         # Reconstruct the complete raw log from all persisted pieces.
#         training_df = self.logger.finalize_shards(
#             log_parts_dir,
#             log_path,
#         )

#         print(f"Training completed. Policy saved to {policy_path}")
#         print(f"Complete training log saved to {log_path}")

#         return training_df


## Main

### VideoRecorder Class

In [110]:
from matplotlib.animation import FFMpegWriter, writers


class AOVideoRecorder:
    """Optional low-overhead MP4 recorder for the final AO closed loop.

    The recorder only reads HCIPy/NumPy quantities that are already on CPU
    (PSF, long-exposure PSF, DM commands and Strehl history). It therefore does
    not add extra CUDA -> CPU transfers when the RL actor runs on the GPU.

    To limit plotting/encoding overhead, a frame is generated only every
    ``frame_every`` AO iterations and all Matplotlib artists are reused.
    """

    def __init__(self, path, *, fps=10, frame_every=5, dpi=100,
                 total_steps=None, mode_label="AO"):
        self.path = str(path)
        self.fps = int(fps)
        self.frame_every = max(1, int(frame_every))
        self.dpi = int(dpi)
        self.total_steps = total_steps
        self.mode_label = str(mode_label)

        if not writers.is_available("ffmpeg"):
            raise RuntimeError(
                "FFmpeg is not available to Matplotlib. Install ffmpeg or run "
                "run_AO_system(..., save_video=False)."
            )

        parent = os.path.dirname(os.path.abspath(self.path))
        if parent:
            os.makedirs(parent, exist_ok=True)

        self.fig = None
        self.writer = None
        self.im_psf = None
        self.im_long = None
        self.im_command = None
        self.sr_line = None
        self.ax_sr = None
        self.frame_count = 0

    @staticmethod
    def _as_2d(array):
        """Convert an HCIPy Field / NumPy array to a plottable 2-D array."""
        if hasattr(array, "shaped"):
            arr = np.asarray(array.shaped)
        else:
            arr = np.asarray(array)

        if arr.ndim == 1:
            side = int(round(np.sqrt(arr.size)))
            if side * side == arr.size:
                arr = arr.reshape(side, side)
            else:
                arr = arr.reshape(1, -1)
        return arr

    @classmethod
    def _log_psf(cls, psf, reference_peak):
        arr = cls._as_2d(psf).astype(float, copy=False)
        norm = max(float(reference_peak), np.finfo(float).tiny)
        # Fixed logarithmic scale makes frames directly comparable in time.
        return np.log10(np.clip(arr / norm, 1e-8, None))

    @classmethod
    def _command_map_nm(cls, command):
        return cls._as_2d(np.asarray(command) * 1e9)

    def _initialize(self, env, command):
        self.fig, axes = plt.subplots(2, 2, figsize=(11, 8))
        ax_psf, ax_long, ax_cmd, self.ax_sr = axes.ravel()

        psf_log = self._log_psf(env.psf, env.norm)
        long_log = self._log_psf(env.long_exposure, env.norm)
        cmd_nm = self._command_map_nm(command)

        self.im_psf = ax_psf.imshow(
            psf_log, origin="lower", vmin=-6, vmax=0, cmap="inferno"
        )
        ax_psf.set_title("Instantaneous PSF [log10 / peak]")
        ax_psf.set_axis_off()
        self.fig.colorbar(self.im_psf, ax=ax_psf, fraction=0.046, pad=0.04)

        self.im_long = ax_long.imshow(
            long_log, origin="lower", vmin=-6, vmax=0, cmap="inferno"
        )
        ax_long.set_title("Long-exposure PSF [log10 / peak]")
        ax_long.set_axis_off()
        self.fig.colorbar(self.im_long, ax=ax_long, fraction=0.046, pad=0.04)

        max_command_nm = max(float(env.max_command) * 1e9, np.finfo(float).eps)
        self.im_command = ax_cmd.imshow(
            cmd_nm,
            origin="lower",
            vmin=-max_command_nm,
            vmax=max_command_nm,
            cmap="RdBu_r",
        )
        ax_cmd.set_title("DM command [nm]")
        ax_cmd.set_axis_off()
        self.fig.colorbar(self.im_command, ax=ax_cmd, fraction=0.046, pad=0.04)

        (self.sr_line,) = self.ax_sr.plot([], [], linewidth=1.5)
        self.ax_sr.set_title("Instantaneous Strehl ratio")
        self.ax_sr.set_xlabel("AO iteration")
        self.ax_sr.set_ylabel("SR")
        self.ax_sr.set_ylim(0.0, 1.05)
        xmax = max(1, int(self.total_steps) - 1) if self.total_steps is not None else 1
        self.ax_sr.set_xlim(0, xmax)
        self.ax_sr.grid(alpha=0.25)

        self.fig.tight_layout(rect=[0, 0, 1, 0.95])

        self.writer = FFMpegWriter(
            fps=self.fps,
            codec="libx264",
            metadata={"title": f"Adaptive Optics - {self.mode_label}"},
        )
        self.writer.setup(self.fig, self.path, dpi=self.dpi)

    def add_frame(self, env, command, timestep, le_strehl=np.nan):
        """Encode one frame when ``timestep`` matches the requested cadence."""
        if timestep % self.frame_every != 0:
            return

        if self.fig is None:
            self._initialize(env, command)

        self.im_psf.set_data(self._log_psf(env.psf, env.norm))
        self.im_long.set_data(self._log_psf(env.long_exposure, env.norm))
        self.im_command.set_data(self._command_map_nm(command))

        sr = np.asarray(env.strehl_list, dtype=float)
        x = np.arange(sr.size)
        self.sr_line.set_data(x, sr)
        if self.total_steps is None:
            self.ax_sr.set_xlim(0, max(1, sr.size - 1))

        current_sr = sr[-1] if sr.size else np.nan
        # Reuse the value already computed by the test logger when available.
        # Only compute it here if the logger cadence did not provide it for this frame.
        le_sr = le_strehl
        if not np.isfinite(le_sr):
            le_sr = env.get_long_exposure_strehl()
        le_text = "n/a" if not np.isfinite(le_sr) else f"{100 * le_sr:.1f}%"
        self.fig.suptitle(
            f"{self.mode_label} | step {timestep} | "
            f"SR={100 * current_sr:.1f}% | LE SR={le_text}"
        )

        self.writer.grab_frame()
        self.frame_count += 1

    def close(self):
        if self.writer is not None:
            self.writer.finish()
            self.writer = None
        if self.fig is not None:
            plt.close(self.fig)
            self.fig = None
        if self.frame_count > 0:
            print(f"Video saved to {self.path} ({self.frame_count} frames)")


**Video recording and performance.** The final AO test loop can optionally save MP4 videos for both the RL controller and the classical integrator. The recorder uses only quantities already on the CPU (HCIPy PSFs, DM commands and Strehl history), so it does not create additional CUDA transfers. Rendering and FFmpeg encoding are nevertheless CPU/I/O work; use `SAVE_TEST_VIDEOS = False` for pure performance benchmarks, or increase `VIDEO_EVERY` to reduce the recording overhead.


**Checkpoints and deterministic evaluation.** Training actions remain stochastic as required by SAC. Evaluation episodes and the final RL test use `tanh(mean(s))`. A resumable checkpoint is saved only every `checkpoint_every` TRAIN episodes (20 by default), overwriting `latest_checkpoint.pth`. To bound I/O and file size, the default checkpoint stores the 20,000 most recent replay transitions; set `checkpoint_replay_limit <= 0` to serialize the full replay buffer.

**Final logging strategy.** Instantaneous Strehl and DM command statistics are retained at every AO step. Optimization statistics are sampled every `log_gradient_every` updates and LE Strehl every `le_strehl_every` environment steps, with the final LE value forced at the end of every episode/test. During training, buffered records are written only at checkpoint boundaries into persistent CSV shards and cleared from RAM. The `.pth` checkpoint therefore no longer duplicates the full logging history. At the end, shards are merged automatically into the complete `training_log.csv` and summary CSV files.

### run_AO_system Function

In [111]:
def run_AO_system(
    path,
    args,
    rl: bool,
    layer_seed=None,
    save_video:bool =None,
    video_path=None,
):
    # rl=None,
    # logger=None,
    # *,
    # save_video=None,
    # video_path=None,
    # video_every=None,
    # video_fps=None,
    # video_dpi=None,

    """Run one final AO test episode with RL or the classical integrator.

    The RL policy is deterministic during this final test: the actor uses
    tanh(mean(s)) rather than sampling from its Gaussian policy.

    Video recording is optional and CPU-side. When disabled (the default), no
    Matplotlib/FFmpeg work is performed in the AO loop. When enabled, only
    HCIPy/NumPy quantities already resident on CPU are visualized, so there are
    no additional CUDA -> CPU transfers beyond the action transfer required by
    the physical HCIPy environment itself.

    Parameters
    ----------
    save_video : bool or None
        Enable MP4 recording. If None, use ``args.save_video``.
    video_path : str or None
        Output MP4 path. If omitted, an RL/classical filename is chosen.
    video_every : int or None
        Record one frame every N AO iterations.
    video_fps : int or None
        Output movie frame rate.
    video_dpi : int or None
        Matplotlib rendering DPI.

    Returns
    -------
    env : AOS
        Final AO environment.
    logger : RLLogger
        Per-step test statistics IFF logger is activated
    """

    # save_video=args.save_video and args.save_log #can save video only when logger is activated
    # if args.save_video and not args.save_log:
    #   print("WARNING: Video disabled because logger is disabled.")


    device = resolve_device(getattr(args, "device", "auto"))

    logger=None
    if args.save_log:
        logger = RLLogger(
            gradient_log_every=args.log_gradient_every,
            le_strehl_every=args.le_strehl_every,
        )

    test_mode = "RL" if rl else "CLASSICAL"

    print(
        f"\n[TEST | {test_mode}] Starting final AO test "
        f"(episode 1/1, {args.system_steps} steps)"
    )

    actor = None
    if rl:
        actor = Actor().to(device)
        actor.load_state_dict(torch.load(path, map_location=device))
        actor.eval()

    env = AOS(args, layer_seed)

    # [v2] load the normalisation scales that go with the policy
    if rl:
        scales_path = path + ".scales.json"
        if os.path.isfile(scales_path):
            import json
            with open(scales_path) as f:
                _scales = json.load(f)
            env.rec_scale = float(_scales["rec_scale"])
            env.cmd_scale = float(_scales["cmd_scale"])
            env.max_action = args.coeff_max_action * env.rec_scale
        else:
            print(f"WARNING: {scales_path} not found: default scales (max_command) in use; "
                  "the policy inputs will not be consistently normalised.")

    for _ in range(args.warm_up_steps):
        env.compute_reconstruction()
        env.apply_DM_command()

    recorder = None
    if save_video:

      # if args.video_path is None:
      #   args.video_path = "AO_with_RL.mp4" if rl else "AO_without_RL.mp4"

      video_path = (
      args.video_path
      if args.video_path is not None
      else video_path #("AO_with_RL.mp4" if rl else "AO_without_RL.mp4")
      )

      recorder = AOVideoRecorder(
          video_path,
          fps=args.video_fps,
          frame_every=args.video_every,
          dpi=args.video_dpi,
          total_steps=args.system_steps,
          mode_label="AO + RL" if rl else "Classical AO",
      )

    previous_command = None

    try:
        for timestep in range(args.system_steps):
            env.compute_reconstruction()

            if rl:
                # Final RL test is deterministic: a = tanh(mean(s)).
                rl_control_step(
                    env, actor, device=device, deterministic=True
                )
            else:
                env.apply_DM_command()

            command = np.asarray(env.commands[-1]).copy()

            le_every = args.le_strehl_every
            if (
                (le_every > 0 and timestep % le_every == 0)
                or timestep == args.system_steps - 1
            ):
                # Always keep the true final long-exposure Strehl.
                le_strehl = env.get_long_exposure_strehl()
            else:
                le_strehl = np.nan

            if args.save_log:
              logger.log_environment_step(
                  phase="test",
                  episode=0,
                  episode_step=timestep,
                  global_env_step=timestep,
                  strehl=env.strehl_list[-1],
                  le_strehl=le_strehl,
                  command=command,
                  max_command=env.max_command,
                  previous_command=previous_command,
              )
            previous_command = command

            if recorder is not None:
                recorder.add_frame(env, command, timestep, le_strehl=le_strehl)
    finally:
        if recorder is not None:
            recorder.close()

    print(
        f"[TEST | {test_mode}] Completed final AO test "
        f"(episode 1/1, {args.system_steps} steps)"
    )

    return env, logger


### Main

In [112]:

import os
import shutil

IS_COLAB = False

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IS_COLAB = True
except Exception as e:
    print(f"Impossibile montare Google Drive ({type(e).__name__}: {e}). Salvo i risultati localmente.")
    IS_COLAB = False

if IS_COLAB:
    # Persistent but slower storage: use only for rare checkpoints/log shards
    # and final artifacts.
    PERSISTENT_DIR = "/content/drive/MyDrive/RL_AO"
    # Fast ephemeral VM storage: use for video encoding and temporary files.
    WORK_DIR = "/content/RL_AO_work"
else:
    PERSISTENT_DIR = os.path.abspath("RL_AO_results")
    WORK_DIR = os.path.abspath("RL_AO_work")

os.makedirs(PERSISTENT_DIR, exist_ok=True)
os.makedirs(WORK_DIR, exist_ok=True)

args.checkpoint_dir = os.path.join(PERSISTENT_DIR, "checkpoints")

TRAINING_LOG_PATH = os.path.join(
    PERSISTENT_DIR,
    "training_log.csv",
)
POLICY_PATH = os.path.join(
    PERSISTENT_DIR,
    "sac_policy.pth",
)

# To resume after a Colab interruption:
# RESUME_FROM = os.path.join(
#     PERSISTENT_DIR, "checkpoints", "latest_checkpoint.pth"
# )
RESUME_FROM = None
# if not os.path.isfile(RESUME_FROM):  # [v2] first run: nothing to resume
#     RESUME_FROM = None


# ------------------------------------------------------------------
# FINAL TESTS
# ------------------------------------------------------------------
print("\n" + "=" * 72)
print("STARTING FINAL AO TESTS")
print("=" * 72)

# The deterministic RL controller and the classical controller are each
# evaluated on the same three atmospheric realisations (paired seeds).
NUM_TEST_REPEATS = args.num_test_repeats #1 #3
TEST_BASE_SEED = 1234

# During the tests, LE Strehl is evaluated at every AO iteration so that
# mean, variance and standard deviation can be computed step by step.
#
# To avoid tripling the FFmpeg overhead, only the first repetition of each
# controller is recorded as video.
RUN_CLASSICAL_REFERENCE = args.run_classical_reference


def aggregate_le_strehl_per_iteration(df):
    """Compute LE SR statistics across test repetitions at each AO iteration."""
    le_df = df[
        (df["type"] == "env")
        & np.isfinite(df["le_strehl"])
    ].copy()

    return (
        le_df
        .groupby("episode_step", as_index=False)
        .agg(
            le_strehl_mean=("le_strehl", "mean"),
            le_strehl_variance=("le_strehl", "var"),
            le_strehl_std=("le_strehl", "std"),
        )
    )



# ------------------------------------------------------------------
# CLASSICAL TEST: 3 repetitions
# ------------------------------------------------------------------
test_classical_df = None
test_classical_files = None
test_classical_le_per_iteration = None

if RUN_CLASSICAL_REFERENCE:
    classical_test_runs = []
    test_classical_env = None

    classical_video_local = os.path.join(
        WORK_DIR,
        "AO_without_RL.mp4",
    )

    for repeat_idx in range(NUM_TEST_REPEATS):
        # Reuse this exact atmospheric realisation later for the RL test.
        #np.random.seed(TEST_BASE_SEED + repeat_idx)
        test_seed = TEST_BASE_SEED + repeat_idx
        print(
            f"\n[TEST | CLASSICAL] Repetition "
            f"{repeat_idx + 1}/{NUM_TEST_REPEATS}"
        )

        # LE SR every AO iteration for the final repeated tests.
          # repeat_logger = RLLogger(
          #     gradient_log_every=getattr(args, "log_gradient_every", 10),
          #     le_strehl_every=1,
          # )

        test_classical_env, repeat_logger = run_AO_system(  # (env restituito dalla funzione non è mai riutilizzato nel main)
            POLICY_PATH,  # ignored when rl=False
            args,
            rl = False,
            layer_seed = test_seed,
            save_video=(args.save_video and repeat_idx == 0),
            video_path=classical_video_local,
        )
      # rl=False,
      # logger=repeat_logger,
      # save_video=(
      #     SAVE_TEST_VIDEOS
      #     and repeat_idx == 0
      # ),
      # video_path=classical_video_local,
      # video_every=VIDEO_EVERY,
      # video_fps=VIDEO_FPS,

        if args.save_log:
          repeat_df = repeat_logger.dataframe().copy()
          repeat_df["repeat"] = repeat_idx + 1
          repeat_df["episode"] = repeat_idx + 1

          classical_test_runs.append(repeat_df)

    if args.save_log:
      test_classical_df = pd.concat(
          classical_test_runs,
          ignore_index=True,
      )

      # Save the complete raw statistics of all three classical repetitions.
      test_classical_files = RLLogger.save_statistics_bundle(
            test_classical_df,
            PERSISTENT_DIR,
            "test_classical",
      )

      # Mean, variance and std of LE SR at every iteration.
      test_classical_le_per_iteration = aggregate_le_strehl_per_iteration(
          test_classical_df
      )

      CLASSICAL_LE_STATS_PATH = os.path.join(
          PERSISTENT_DIR,
          "test_classical_LE_SR_per_iteration.csv",
      )

      RLLogger._atomic_csv_write(
          test_classical_le_per_iteration,
          CLASSICAL_LE_STATS_PATH,
      )

    if args.save_video:
      if os.path.isfile(classical_video_local):
          shutil.copy2(
              classical_video_local,
              os.path.join(PERSISTENT_DIR, "AO_without_RL.mp4"),
          )

# ------------------------------------------------------------------
# TRAINING
# ------------------------------------------------------------------
print("\n" + "=" * 72)
print("STARTING SAC TRAINING / EVALUATION")
print("=" * 72)

sac = SAC(args)

training_df = sac.train_eval(
    policy_path=POLICY_PATH,
    log_path=TRAINING_LOG_PATH,
    resume_from=RESUME_FROM,
)

if args.save_log:
  training_files = RLLogger.save_statistics_bundle(
      training_df,
      PERSISTENT_DIR,
      "training",
  )


# # ------------------------------------------------------------------
# # RL TEST: 3 repetitions
# # ------------------------------------------------------------------
rl_test_runs = []
test_rl_env = None

rl_video_local = os.path.join(
    WORK_DIR,
    "AO_with_RL.mp4",
)

for repeat_idx in range(NUM_TEST_REPEATS):
    # Same seed as the matching classical repetition: paired comparison.
    #np.random.seed(TEST_BASE_SEED + repeat_idx)
    test_seed = np.random.seed(TEST_BASE_SEED + repeat_idx)
    print(
        f"\n[TEST | RL] Repetition "
        f"{repeat_idx + 1}/{NUM_TEST_REPEATS}"
    )

    # LE SR every AO iteration for the final repeated tests.
    # if args.save_log:
      # repeat_logger = RLLogger(
      #     gradient_log_every=getattr(args, "log_gradient_every", 10),
      #     le_strehl_every=1,
      # )


    test_rl_env, repeat_logger = run_AO_system( # (env restituito dalla funzione non è mai riutilizzato nel main)
        POLICY_PATH,
        args,
        rl = True,
        layer_seed = test_seed,
        save_video=(args.save_video and repeat_idx == 0),
        video_path=rl_video_local,
    )
      # rl=True,
      # logger=repeat_logger,
      # save_video=(
      #     SAVE_TEST_VIDEOS
      #     and repeat_idx == 0
      # ),
      # video_path=rl_video_local,
      # video_every=VIDEO_EVERY,
      # video_fps=VIDEO_FPS,

    if args.save_log:
      repeat_df = repeat_logger.dataframe().copy()

      # Reuse the existing "episode" field to distinguish the 3 test repetitions.
      repeat_df["repeat"] = repeat_idx + 1
      repeat_df["episode"] = repeat_idx + 1

      rl_test_runs.append(repeat_df)

if args.save_log:
  test_rl_df = pd.concat(
      rl_test_runs,
      ignore_index=True,
  )

  # Save the complete raw statistics of all three RL repetitions.
  test_rl_files = RLLogger.save_statistics_bundle(
      test_rl_df,
      PERSISTENT_DIR,
      "test_RL",
  )

  # Mean, variance and std of LE SR at every iteration.
  test_rl_le_per_iteration = aggregate_le_strehl_per_iteration(
      test_rl_df
  )

  RL_LE_STATS_PATH = os.path.join(
      PERSISTENT_DIR,
      "test_RL_LE_SR_per_iteration.csv",
  )

  RLLogger._atomic_csv_write(
      test_rl_le_per_iteration,
      RL_LE_STATS_PATH,
  )

if args.save_video:
  if os.path.isfile(rl_video_local):
      shutil.copy2(
          rl_video_local,
          os.path.join(PERSISTENT_DIR, "AO_with_RL.mp4"),
      )


# ------------------------------------------------------------------
# LE STREHL COMPARISON PLOT
# ------------------------------------------------------------------
if args.save_log and test_classical_le_per_iteration is not None:
    fig, ax = plt.subplots(figsize=(10, 6))

    rl_x = test_rl_le_per_iteration["episode_step"].to_numpy()
    rl_mean = test_rl_le_per_iteration["le_strehl_mean"].to_numpy()
    rl_std = test_rl_le_per_iteration["le_strehl_std"].to_numpy()

    classical_x = test_classical_le_per_iteration["episode_step"].to_numpy()
    classical_mean = test_classical_le_per_iteration["le_strehl_mean"].to_numpy()
    classical_std = test_classical_le_per_iteration["le_strehl_std"].to_numpy()

    ax.plot(
        rl_x,
        rl_mean,
        label="RL",
    )
    ax.fill_between(
        rl_x,
        rl_mean - rl_std,
        rl_mean + rl_std,
        alpha=0.2,
    )

    ax.plot(
        classical_x,
        classical_mean,
        label="Classical",
    )
    ax.fill_between(
        classical_x,
        classical_mean - classical_std,
        classical_mean + classical_std,
        alpha=0.2,
    )

    ax.set_xlabel("AO iteration")
    ax.set_ylabel("Long-exposure Strehl ratio")
    ax.set_title(
        f"Long-exposure Strehl ratio: mean ± std "
        f"over {NUM_TEST_REPEATS} repetitions"
    )
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()

    LE_PLOT_PATH = os.path.join(
        PERSISTENT_DIR,
        "LE_SR_RL_vs_classical_mean_std.png",
    )

    fig.savefig(
        LE_PLOT_PATH,
        dpi=200,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)


# ------------------------------------------------------------------
# READY-TO-USE SUMMARY TABLES
# ------------------------------------------------------------------
if args.save_log:

  training_strehl_summary = RLLogger.strehl_summary_from_df(
      training_df
  )
  training_le_strehl_by_episode = RLLogger.le_strehl_episode_from_df(
      training_df
  )
  training_le_strehl_summary = RLLogger.le_strehl_summary_from_df(
      training_df
  )

  test_rl_strehl_summary = RLLogger.strehl_summary_from_df(
      test_rl_df,
      phase="test",
  )
  test_rl_le_strehl = RLLogger.le_strehl_episode_from_df(
      test_rl_df,
      phase="test",
  )

  print("\nTraining Strehl summary:")
  display(training_strehl_summary)

  print("\nTraining final LE Strehl by episode:")
  display(training_le_strehl_by_episode)

  print("\nTraining LE Strehl aggregate summary:")
  display(training_le_strehl_summary)

  print("\nRL test Strehl summary (3 repetitions):")
  display(test_rl_strehl_summary)

  print("\nRL test final LE Strehl by repetition:")
  display(test_rl_le_strehl)

  print("\nRL LE Strehl mean/variance/std per AO iteration:")
  display(test_rl_le_per_iteration)

  if test_classical_df is not None:
      print("\nClassical test Strehl summary (3 repetitions):")
      display(
          RLLogger.strehl_summary_from_df(
              test_classical_df,
              phase="test",
          )
      )

      print("\nClassical test final LE Strehl by repetition:")
      display(
          RLLogger.le_strehl_episode_from_df(
              test_classical_df,
              phase="test",
          )
      )

      print("\nClassical LE Strehl mean/variance/std per AO iteration:")
      display(test_classical_le_per_iteration)

  print(f"\nAll persistent results are in: {PERSISTENT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

STARTING FINAL AO TESTS

STARTING SAC TRAINING / EVALUATION
r0   = 16.8cm
Actuators spacing: 162.50 cm


  0%|          | 0/16 [00:00<?, ?it/s]

Running on cpu

STARTING RMS CALIBRATION
___Strehl Ratio over 10 iterations___
SR: 0.1%
LE SR: 0.0%
DM RMS:    270.0 nm
___Strehl Ratio over 20 iterations___
SR: 0.1%
LE SR: 0.1%
DM RMS:    281.6 nm
___Strehl Ratio over 30 iterations___
SR: 0.4%
LE SR: 0.1%
DM RMS:    291.5 nm
___Strehl Ratio over 40 iterations___
SR: 0.6%
LE SR: 0.3%
DM RMS:    304.5 nm
___Strehl Ratio over 50 iterations___
SR: 0.5%
LE SR: 0.3%
DM RMS:    318.4 nm
___Strehl Ratio over 60 iterations___
SR: 0.4%
LE SR: 0.4%
DM RMS:    322.0 nm
___Strehl Ratio over 70 iterations___
SR: 0.5%
LE SR: 0.4%
DM RMS:    320.7 nm
___Strehl Ratio over 80 iterations___
SR: 0.8%
LE SR: 0.4%
DM RMS:    313.7 nm
___Strehl Ratio over 90 iterations___
SR: 1.2%
LE SR: 0.5%
DM RMS:    300.5 nm
___Strehl Ratio over 100 iterations___
SR: 1.8%
LE SR: 0.6%
DM RMS:    296.1 nm
___Strehl Ratio over 110 iterations___
SR: 1.9%
LE SR: 0.7%
DM RMS:    296.5 nm
___Strehl Ratio over 120 iterations___
SR: 1.2%
LE SR: 0.8%
DM RMS:    286.9 nm
___Streh

  0%|          | 0/7 [00:00<?, ?it/s]


[TRAIN | RL] Episode 1/7 (training #1/5)
___Strehl Ratio over 10 iterations___
SR: 1.9%
LE SR: 1.7%
DM RMS:    272.2 nm
[TRAIN | RL] Completed training episode 1/5

[TRAIN | RL] Episode 2/7 (training #2/5)
___Strehl Ratio over 10 iterations___
SR: 1.7%
LE SR: 1.1%
DM RMS:    547.7 nm
[TRAIN | RL] Completed training episode 2/5

[EVAL | RL] Episode 3/7 (evaluation #1)
___Strehl Ratio over 10 iterations___
SR: 2.6%
LE SR: 1.9%
DM RMS:    220.6 nm
[EVAL | RL] final long-exposure Strehl = 0.0193

[TRAIN | RL] Episode 4/7 (training #3/5)
___Strehl Ratio over 10 iterations___
SR: 1.9%
LE SR: 1.3%
DM RMS:    314.2 nm
[TRAIN | RL] Completed training episode 3/5

[TRAIN | RL] Episode 5/7 (training #4/5)
___Strehl Ratio over 10 iterations___
SR: 0.1%
LE SR: 0.3%
DM RMS:    436.1 nm
[TRAIN | RL] Completed training episode 4/5

[EVAL | RL] Episode 6/7 (evaluation #2)
___Strehl Ratio over 10 iterations___
SR: 0.5%
LE SR: 0.6%
DM RMS:    257.5 nm
[EVAL | RL] final long-exposure Strehl = 0.0061

[TR

  0%|          | 0/16 [00:00<?, ?it/s]

___Strehl Ratio over 10 iterations___
SR: 0.2%
LE SR: 0.1%
DM RMS:    292.9 nm
___Strehl Ratio over 20 iterations___
SR: 0.8%
LE SR: 0.5%
DM RMS:    324.1 nm
___Strehl Ratio over 30 iterations___
SR: 0.5%
LE SR: 0.6%
DM RMS:    328.6 nm
___Strehl Ratio over 40 iterations___
SR: 0.6%
LE SR: 0.5%
DM RMS:    321.8 nm
___Strehl Ratio over 50 iterations___
SR: 0.8%
LE SR: 0.6%
DM RMS:    304.9 nm
___Strehl Ratio over 60 iterations___
SR: 0.8%
LE SR: 0.6%
DM RMS:    288.9 nm
___Strehl Ratio over 70 iterations___
SR: 0.5%
LE SR: 0.6%
DM RMS:    281.4 nm
___Strehl Ratio over 80 iterations___
SR: 0.4%
LE SR: 0.6%
DM RMS:    287.2 nm
___Strehl Ratio over 90 iterations___
SR: 0.9%
LE SR: 0.6%
DM RMS:    321.3 nm
___Strehl Ratio over 100 iterations___
SR: 1.2%
LE SR: 0.6%
DM RMS:    377.7 nm
[TEST | RL] Completed final AO test (episode 1/1, 100 steps)
